# RPKClust: self-contained runner

This notebook embeds every Python source file from the project. Run the setup cell first; it reconstructs an isolated temporary package so the remaining cells work without the original `.py` files.

In [1]:
%pip install -q numpy scipy scikit-learn matplotlib pandas seaborn tabulate psutil scapy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Embedded project sources. This dictionary is generated from the repository.
SOURCES = {'main.py': 'import os\nimport sys\n\n# Ensure local packages and modules resolve correctly\nsys.path.append(os.path.dirname(os.path.abspath(__file__)))\n\nfrom datasets.generate_data import *\nfrom datasets.stress_generator import BinaryProtocolStressGenerator\nfrom datasets.dataset_loader import PcapDatasetLoader\nfrom temp_evaluator import RPKClustEvaluator\n\n\ndef main():\n    print("==========================================================")\n    print(" Initializing RPKClust Paper-Faithful Evaluation Suite")\n    print("==========================================================\\n")\n\n    evaluator = RPKClustEvaluator(output_dir="results", fig_format="png", dpi=300)\n    generator = GenericDatasetGenerator(seed=3456732456)\n    # 1. Generic Fixed-Offset Region (FOR) Dataset\n    X_for, y_for, metadata = generator.generate_for_dataset(num_messages=1000)\n    evaluator.run_diagnostics(\n        X_for,\n        y_for,\n        dataset_name="Generic FOR",\n        true_boundary=metadata["true_boundary_B"],\n        true_keyword_offset=metadata["true_keyword_offset"],\n        fit_kwargs={"interaction_metadata": metadata["interaction_metadata"]},\n    )\n\n    # 2. Generic Non-Fixed-Offset Region (NFOR TLV) Dataset\n    X_nfor, y_nfor, nfor_metadata = generator.generate_nfor_dataset(num_messages=1000)\n    evaluator.run_diagnostics(\n        X_nfor,\n        y_nfor,\n        dataset_name="Generic NFOR TLV",\n        true_boundary=nfor_metadata["true_boundary_B"],\n        fit_kwargs={"interaction_metadata": nfor_metadata["interaction_metadata"]},\n    )\n\n    # 3. Binary Stress Test Dataset\n    stress_gen = BinaryProtocolStressGenerator(num_messages=1000, noise_level=0.15, seed=4151684516)\n    X_stress, y_stress, stress_metadata = stress_gen.generate_with_metadata()\n    evaluator.run_diagnostics(\n        X_stress,\n        y_stress,\n        dataset_name="Binary Protocol Stress",\n        true_boundary=12,\n        true_keyword_offset=3,\n        fit_kwargs={"interaction_metadata": stress_metadata["interaction_metadata"]},\n    )\n\n    # 4. Real-World PCAP Traffic Dataset (Fetched via PcapDatasetLoader)\n    print("\\n" + "=" * 65)\n    print(" FETCHING & PROCESSING EXTERNAL REAL PCAP DATASET")\n    print("=" * 65)\n    pcap_loader = PcapDatasetLoader()\n    pcap_file = pcap_loader.download_pcap()\n    X_pcap, y_pcap, pcap_metadata = pcap_loader.extract_payloads_with_metadata(pcap_file)\n\n    evaluator.run_diagnostics(\n        X_pcap,\n        y_pcap,\n        dataset_name="Real Network PCAP Dataset",\n        fit_kwargs={"interaction_metadata": pcap_metadata} if pcap_metadata else None,\n    )\n\n    # Export Aggregate Tables and Comparative Visualizations\n    print("\\n" + "=" * 65)\n    print(" GENERATING FINAL PAPER ARTIFACTS")\n    print("=" * 65)\n    evaluator.export_summary_artifacts()\n    print("\\nExecution complete! Check results/tables/ and results/figures/ for output.")\n\n\nif __name__ == "__main__":\n    main()', 'temp_evaluator.py': '"""\nRPKClust Paper-Accurate Evaluator and Artifact Generator (Section 4).\n\nGenerates paper-accurate evaluation results:\n  - Terminal diagnostic output (FOR-NFOR error, Stage 1/2 probabilities, paper metrics).\n  - Markdown / LaTeX-formatted tables saved to results/tables/.\n  - Publication-ready visual diagnostic figures saved to results/figures/.\n"""\n\nimport os\nimport re\nimport time\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom typing import List, Dict, Any, Optional, Tuple\n\nfrom rpkclust import RPKClust\nfrom rpkclust.metrics import (\n    evaluate_clustering,\n    evaluate_boundary,\n    evaluate_keyword_inference,\n    convert_bytes_to_feature_matrix,\n    measure_memory_usage,\n)\n\n\ndef _to_markdown(frame: pd.DataFrame) -> str:\n    """Render a table without requiring pandas\' optional tabulate package."""\n    try:\n        return frame.to_markdown(index=False)\n    except ImportError:\n        headers = [str(column) for column in frame.columns]\n        rows = [[str(value) for value in row] for row in frame.itertuples(index=False, name=None)]\n        widths = [max(len(header), *(len(row[i]) for row in rows)) for i, header in enumerate(headers)]\n        header_row = "| " + " | ".join(header.ljust(widths[i]) for i, header in enumerate(headers)) + " |"\n        separator = "|" + "|".join("-" * (width + 2) for width in widths) + "|"\n        body = ["| " + " | ".join(value.ljust(widths[i]) for i, value in enumerate(row)) + " |" for row in rows]\n        return "\\n".join([header_row, separator, *body])\n\n\nclass RPKClustEvaluator:\n    """\n    Paper-faithful evaluation suite for RPKClust pipeline.\n    Handles benchmarking, reporting, table export, and figure rendering.\n    """\n\n    def __init__(\n        self,\n        output_dir: str = "results",\n        fig_format: str = "png",\n        dpi: int = 300,\n    ):\n        self.output_dir = output_dir\n        self.tables_dir = os.path.join(output_dir, "tables")\n        self.figures_dir = os.path.join(output_dir, "figures")\n        self.fig_format = fig_format\n        self.dpi = dpi\n\n        os.makedirs(self.tables_dir, exist_ok=True)\n        os.makedirs(self.figures_dir, exist_ok=True)\n\n        # Storage for multi-dataset comparative reports\n        self.summary_records: List[Dict[str, Any]] = []\n\n        # Configure matplotlib style for publication-grade plots\n        plt.style.use("seaborn-v0_8-paper" if "seaborn-v0_8-paper" in plt.style.available else "default")\n        plt.rcParams.update({\n            "font.family": "sans-serif",\n            "font.size": 10,\n            "axes.labelsize": 11,\n            "axes.titlesize": 12,\n            "xtick.labelsize": 10,\n            "ytick.labelsize": 10,\n            "legend.fontsize": 9,\n            "figure.titlesize": 13,\n        })\n\n    def run_diagnostics(\n        self,\n        X: List[bytes],\n        y_true: np.ndarray,\n        dataset_name: str = "Dataset",\n        true_boundary: Optional[int] = None,\n        true_keyword_offset: Optional[Any] = None,\n        fit_kwargs: Optional[Dict[str, Any]] = None,\n    ) -> Dict[str, Any]:\n        """\n        Executes a single-dataset diagnostic pass, prints terminal diagnostics,\n        and saves dataset-level tables and figures.\n        """\n        fit_kwargs = fit_kwargs or {}\n        if len(X) != len(y_true):\n            raise ValueError("X and y_true must contain the same number of messages")\n        if not X:\n            raise ValueError("at least one message is required for diagnostics")\n\n        print("\\n" + "=" * 65)\n        print(f"  RPKCLUST PAPER DIAGNOSTIC REPORT: {dataset_name}")\n        print("=" * 65)\n\n        # Memory baseline before run\n        mem_before = measure_memory_usage()\n\n        # Fit Model and measure execution time\n        model = RPKClust()\n        t0 = time.perf_counter()\n        model.fit(X, **fit_kwargs)\n        exec_time = time.perf_counter() - t0\n        mem_after = measure_memory_usage()\n        memory_mb = max(0.0, round(mem_after - mem_before, 2))\n\n        # ---- 1. Boundary Identification Evaluation (Paper Section 4.3)\n        print("\\n[1] FOR-NFOR BOUNDARY IDENTIFICATION (Algorithm 1)")\n        print(f"  Inferred FOR-NFOR Boundary (B) : {model.boundary_B} bytes")\n        print(f"  Execution Time               : {exec_time:.4f} seconds")\n\n        boundary_eval = {}\n        if true_boundary is not None:\n            boundary_eval = evaluate_boundary(true_boundary, model.boundary_B)\n            print(f"  Ground Truth Boundary        : {true_boundary} bytes")\n            print(f"  Boundary Absolute Error      : {boundary_eval[\'Error\']} bytes")\n            print(f"  Boundary Relative Error      : {boundary_eval[\'Error (%)\']:.2f}%")\n\n        # ---- 2. Candidate Generation Breakdown (Paper Algorithms 2 & 3)\n        for_cands = [c for c in model.candidates if c.get("type") == "FOR"]\n        nfor_cands = [c for c in model.candidates if c.get("type") == "NFOR"]\n\n        print("\\n[2] REGION-PARTITIONED CANDIDATE GENERATION")\n        print(f"  FOR Candidates Extracted     : {len(for_cands)}")\n        print(f"  NFOR Candidates Extracted    : {len(nfor_cands)}")\n        print(f"  Total Candidates Evaluated  : {len(model.candidates)}")\n\n        # ---- 3. Two-Stage Bayesian Inference Breakdown (Paper Section 3.6 & 4.4)\n        print("\\n[3] TWO-STAGE BAYESIAN INFERENCE RANKINGS (Top Candidates)")\n        print("-" * 65)\n        print(f"{\'Rank\':<5}{\'Tag\':<22}{\'Type\':<6}{\'p_bit\':<8}{\'p_offset\':<10}{\'Stage1\':<8}{\'Posterior P\':<10}")\n        print("-" * 65)\n\n        top_candidates = model.candidates[:5]\n        for rank, cand in enumerate(top_candidates, 1):\n            c_tag = str(cand.get("tag", "N/A"))[:20]\n            c_type = str(cand.get("type", "FOR"))\n            p_bit = cand.get("p_bit", 0.0)\n            p_offset = cand.get("p_offset", 0.0)\n            p_stage1 = cand.get("stage1_prob", 0.0)\n            p_final = cand.get("prob", 0.0)\n\n            print(f"{rank:<5}{c_tag:<22}{c_type:<6}{p_bit:<8.4f}{p_offset:<10.4f}{p_stage1:<8.4f}{p_final:<10.4f}")\n        print("-" * 65)\n\n        keyword_eval = {}\n        if true_keyword_offset is not None:\n            keyword_eval = evaluate_keyword_inference(model.candidates, true_keyword_offset)\n            print(f"  Ground Truth Keyword Offset  : {true_keyword_offset}")\n            print(f"  Top Inferred Keyword Offset  : {keyword_eval[\'Inferred Keyword Offset\']}")\n            print(f"  Keyword Correctly Ranked #1  : {\'YES\' if keyword_eval[\'Correct\'] else \'NO\'}")\n            print(f"  Keyword Rank                 : {keyword_eval[\'Rank\']}")\n\n        # ---- 4. Clustering Performance Metrics (Paper Section 4.2, Eq. 18-20)\n        y_pred = model.labels_ if model.labels_ is not None else np.zeros(len(X), dtype=int)\n        feature_matrix = convert_bytes_to_feature_matrix(X)\n\n        cluster_metrics = evaluate_clustering(\n            labels_true=y_true,\n            labels_pred=y_pred,\n            feature_matrix=feature_matrix,\n            exec_time=exec_time,\n            memory_mb=memory_mb,\n        )\n\n        print("\\n[4] CLUSTERING PERFORMANCE METRICS (Section 4.2)")\n        print(f"  Homogeneity (Eq. 18)         : {cluster_metrics[\'Homogeneity\']:.4f}")\n        print(f"  Completeness (Eq. 19)        : {cluster_metrics[\'Completeness\']:.4f}")\n        print(f"  V-Measure (Eq. 20)           : {cluster_metrics[\'V-Measure\']:.4f}")\n        print(f"  Adjusted Rand Index (ARI)    : {cluster_metrics[\'ARI\']:.4f}")\n        print(f"  Normalized Mutual Info (NMI) : {cluster_metrics[\'NMI\']:.4f}")\n        print(f"  Clustering Accuracy (Hungarian): {cluster_metrics[\'Clustering Accuracy\']:.4f}")\n        print(f"  Clusters Identified          : {cluster_metrics[\'Clusters Found\']} (True: {len(np.unique(y_true))})")\n        print(f"  Memory Overhead              : {cluster_metrics[\'Memory (MB)\']} MB")\n\n        if model.best_candidate:\n            print(f"\\n  Selected Keyword Tag        : {model.best_candidate.get(\'tag\')}")\n            print(f"  Selected Keyword Type       : {model.best_candidate.get(\'type\')}")\n            print(f"  Selected Keyword Posterior  : {model.best_candidate.get(\'prob\', 0.0):.4f}")\n        print("=" * 65 + "\\n")\n\n        # ---- Aggregate Summary Record\n        summary_record = {\n            "Dataset": dataset_name,\n            "Messages": len(X),\n            "Boundary_Inferred": model.boundary_B,\n            "Boundary_True": true_boundary if true_boundary is not None else "N/A",\n            "Boundary_Error": boundary_eval.get("Error", "N/A"),\n            "FOR_Candidates": len(for_cands),\n            "NFOR_Candidates": len(nfor_cands),\n            "Total_Candidates": len(model.candidates),\n            "Keyword_Correct": keyword_eval.get("Correct", "N/A"),\n            "Keyword_Rank": keyword_eval.get("Rank", "N/A"),\n            "Homogeneity": cluster_metrics["Homogeneity"],\n            "Completeness": cluster_metrics["Completeness"],\n            "V_Measure": cluster_metrics["V-Measure"],\n            "ARI": cluster_metrics["ARI"],\n            "NMI": cluster_metrics["NMI"],\n            "Accuracy": cluster_metrics["Clustering Accuracy"],\n            "Clusters_Found": cluster_metrics["Clusters Found"],\n            "Time_s": cluster_metrics["Execution Time (s)"],\n            "Memory_MB": cluster_metrics["Memory (MB)"],\n        }\n        self.summary_records.append(summary_record)\n\n        # Export dataset-specific plots and candidate ranking table\n        clean_ds_name = re.sub(r"[^a-z0-9_-]+", "_", dataset_name.lower()).strip("_-") or "dataset"\n        self._export_candidate_table(model.candidates, clean_ds_name)\n        self._generate_candidate_probability_figure(model.candidates, dataset_name, clean_ds_name)\n\n        return summary_record\n\n    def export_summary_artifacts(self) -> Tuple[str, str]:\n        """\n        Exports aggregate results across all evaluated datasets to Markdown tables\n        and comparative figures.\n        """\n        if not self.summary_records:\n            print("No evaluation records available to export.")\n            return "", ""\n\n        df_summary = pd.DataFrame(self.summary_records)\n\n        # 1. Export Aggregate Paper Table (Markdown & CSV)\n        csv_path = os.path.join(self.tables_dir, "rpkclust_summary_metrics.csv")\n        md_path = os.path.join(self.tables_dir, "rpkclust_summary_metrics.md")\n\n        df_summary.to_csv(csv_path, index=False)\n\n        # Paper-formatted Markdown Table\n        paper_table_cols = [\n            "Dataset", "Messages", "Boundary_Inferred", "Homogeneity", \n            "Completeness", "V_Measure", "ARI", "Accuracy", "Time_s"\n        ]\n        df_paper = df_summary[paper_table_cols].copy()\n        df_paper.columns = [\n            "Dataset", "Messages", "Boundary (B)", "Homogeneity (h)", \n            "Completeness (c)", "V-Measure (v)", "ARI", "Acc", "Time (s)"\n        ]\n\n        with open(md_path, "w", encoding="utf-8") as f:\n            f.write("# RPKClust Paper Evaluation Summary Table\\n\\n")\n            f.write(_to_markdown(df_paper))\n            f.write("\\n")\n\n        print(f"[Artifact Export] Aggregated summary table written to:\\n  - {md_path}\\n  - {csv_path}")\n\n        # 2. Generate Comparative Figure Across Datasets\n        fig_path = self._generate_comparative_metrics_figure(df_summary)\n\n        return md_path, fig_path\n\n    def _export_candidate_table(self, candidates: List[Dict[str, Any]], dataset_key: str):\n        """Exports top candidate ranking breakdown to Markdown."""\n        rows = []\n        for rank, c in enumerate(candidates[:10], 1):\n            rows.append({\n                "Rank": rank,\n                "Tag": c.get("tag", "N/A"),\n                "Type": c.get("type", "N/A"),\n                "Offset": c.get("offset", "N/A"),\n                "p_bit": round(c.get("p_bit", 0.0), 4),\n                "p_offset": round(c.get("p_offset", 0.0), 4),\n                "Stage1_Prob": round(c.get("stage1_prob", 0.0), 4),\n                "Posterior_P": round(c.get("prob", 0.0), 4),\n            })\n\n        df_cands = pd.DataFrame(rows)\n        md_path = os.path.join(self.tables_dir, f"candidates_{dataset_key}.md")\n        with open(md_path, "w", encoding="utf-8") as f:\n            f.write(f"# Top Candidate Keyword Rankings: {dataset_key}\\n\\n")\n            f.write(_to_markdown(df_cands))\n            f.write("\\n")\n\n    def _generate_candidate_probability_figure(\n        self,\n        candidates: List[Dict[str, Any]],\n        dataset_name: str,\n        dataset_key: str,\n    ):\n        """Generates a publication-grade bar chart comparing Stage 1 vs Stage 2 probabilities."""\n        if not candidates:\n            return\n\n        top_cands = candidates[:8]\n        tags = [c.get("tag", f"Cand {i}")[:15] for i, c in enumerate(top_cands)]\n        p_stage1 = [c.get("stage1_prob", 0.0) for c in top_cands]\n        p_posterior = [c.get("prob", 0.0) for c in top_cands]\n\n        x = np.arange(len(tags))\n        width = 0.35\n\n        fig, ax = plt.subplots(figsize=(8, 4.5))\n        rects1 = ax.bar(x - width/2, p_stage1, width, label="Stage 1 (p_f)", color="#4C72B0")\n        rects2 = ax.bar(x + width/2, p_posterior, width, label="Stage 2 Posterior P(K=1|D)", color="#55A868")\n\n        ax.set_ylabel("Probability Score")\n        ax.set_title(f"RPKClust Inference Stage Comparison — {dataset_name}")\n        ax.set_xticks(x)\n        ax.set_xticklabels(tags, rotation=25, ha="right")\n        ax.set_ylim(0.0, 1.05)\n        ax.legend()\n        ax.grid(axis="y", linestyle="--", alpha=0.5)\n\n        plt.tight_layout()\n        out_file = os.path.join(self.figures_dir, f"stage_inference_{dataset_key}.{self.fig_format}")\n        plt.savefig(out_file, dpi=self.dpi)\n        plt.close()\n\n    def _generate_comparative_metrics_figure(self, df_summary: pd.DataFrame) -> str:\n        """Generates a multi-metric comparative bar chart across all evaluated datasets."""\n        datasets = df_summary["Dataset"].tolist()\n        metrics = ["Homogeneity", "Completeness", "V_Measure", "ARI"]\n\n        x = np.arange(len(datasets))\n        width = 0.18\n\n        fig, ax = plt.subplots(figsize=(9, 5))\n        colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]\n\n        for i, metric in enumerate(metrics):\n            scores = df_summary[metric].values\n            ax.bar(x + (i - 1.5) * width, scores, width, label=metric, color=colors[i])\n\n        ax.set_ylabel("Score [0.0 - 1.0]")\n        ax.set_title("RPKClust Clustering Performance Across Benchmark Datasets")\n        ax.set_xticks(x)\n        ax.set_xticklabels(datasets, rotation=15, ha="right")\n        ax.set_ylim(0.0, 1.05)\n        ax.legend(loc="lower right")\n        ax.grid(axis="y", linestyle="--", alpha=0.5)\n\n        plt.tight_layout()\n        out_file = os.path.join(self.figures_dir, f"rpkclust_benchmark_comparison.{self.fig_format}")\n        plt.savefig(out_file, dpi=self.dpi)\n        plt.close()\n\n        print(f"[Artifact Export] Comparative benchmark plot saved to:\\n  - {out_file}")\n        return out_file', 'rpkclust/__init__.py': 'from .rpkclust import RPKClust\n\n__all__ = ["RPKClust"]', 'rpkclust/constraints.py': '"""\nNetplier-style Clustering Constraints for RPKClust Stage 1.\n\nRPKClust Section 3.6: "we use the four constraints proposed in\nNetplier: message similarity constraint, remote coupling constraint,\nstructural consistency constraint, and dimensional constraint."\n\nThe RPKClust paper does not redefine these constraints — it references\nNetplier [29] directly. This implementation follows Netplier\'s definitions\nadapted for RPKClust\'s region-partitioned (non-MSA) pipeline.\n\nNetplier reference: Ye et al., "NETPLIER: Probabilistic Network Protocol\nReverse Engineering from Message Traces," NDSS 2021.\n\nNOTE: Because the paper does not provide exact formulas, these are labeled\nas "Netplier-style approximations aligned with RPKClust\'s described inputs."\n"""\n\nimport numpy as np\nimport warnings\nfrom typing import List, Dict, Any, Optional, Tuple\nfrom collections import defaultdict\n\n\nclass ClusteringConstraints:\n    """\n    Four clustering constraints from Netplier, adapted for RPKClust.\n\n    Each constraint returns a float in [0, 1] representing the degree\n    of constraint satisfaction for the current clustering.\n    """\n\n    # ==============================================================\n    #  1. Message Similarity Constraint\n    # ==============================================================\n\n    @staticmethod\n    def _byte_similarity(a: bytes, b: bytes) -> float:\n        """\n        Netplier byte-level similarity: number of matching bytes at the\n        same position divided by the maximum message length.\n        Extra bytes in the longer message count as mismatches.\n\n        Netplier: s = (number of same bytes) / (total bytes compared).\n        """\n        max_len = max(len(a), len(b))\n        if max_len == 0:\n            return 1.0\n\n        min_len = min(len(a), len(b))\n        same = sum(1 for i in range(min_len) if a[i] == b[i])\n        return same / max_len\n\n    @staticmethod\n    def message_similarity(\n        labels: np.ndarray,\n        X: List[bytes],\n    ) -> float:\n        """\n        Netplier Message Similarity Constraint.\n\n        Netplier: "messages in the same cluster should have higher\n        similarity than messages in different clusters."\n\n        Computes byte-level similarity (matching_bytes / max_len) for all\n        message pairs. Separates into intra-cluster and inter-cluster\n        score distributions. The constraint is satisfied when intra-\n        cluster scores are consistently higher than inter-cluster scores.\n\n        Returns the probability that the constraint is observed,\n        based on the overlap (false match + false non-match) between\n        the two distributions.\n        """\n        n = len(labels)\n        if len(X) != n:\n            raise ValueError("labels and X must contain the same number of messages")\n        if n < 2:\n            return 0.0\n\n        intra_scores: List[float] = []\n        inter_scores: List[float] = []\n\n        for i in range(n):\n            for j in range(i + 1, n):\n                similarity = ClusteringConstraints._byte_similarity(\n                    X[i], X[j]\n                )\n\n                if labels[i] == labels[j]:\n                    intra_scores.append(similarity)\n                else:\n                    inter_scores.append(similarity)\n\n        if not intra_scores:\n            return 0.0\n        if not inter_scores:\n            # All messages in one cluster — high intra similarity.\n            return float(np.mean(intra_scores))\n\n        intra_arr = np.array(intra_scores)\n        inter_arr = np.array(inter_scores)\n\n        # Netplier: compute false match and false non-match errors.\n        # False match: inter-cluster score >= threshold\n        #   (different types incorrectly grouped together).\n        # False non-match: intra-cluster score <= threshold\n        #   (same type incorrectly split apart).\n        #\n        # Use a threshold at the midpoint between the two means.\n        threshold = (np.mean(intra_arr) + np.mean(inter_arr)) / 2.0\n\n        # False match rate: inter scores above threshold.\n        false_match = np.mean(inter_arr >= threshold)\n        # False non-match rate: intra scores below threshold.\n        false_nonmatch = np.mean(intra_arr <= threshold)\n\n        # Constraint probability = 1 - total error rate.\n        error_rate = (false_match + false_nonmatch) / 2.0\n        p_m = 1.0 - error_rate\n\n        return float(np.clip(p_m, 0.0, 1.0))\n\n    # ==============================================================\n    #  2. Remote Coupling Constraint\n    # ==============================================================\n\n    @staticmethod\n    def remote_coupling(\n        labels: np.ndarray,\n        X: List[bytes],\n        interaction_metadata: Optional[List[Dict[str, Any]]] = None,\n    ) -> float:\n        """\n        Netplier Remote Coupling Constraint.\n\n        Netplier: "client and server clusters should have corresponding\n        relationships." Uses interaction metadata (source/dest IP, ports,\n        timestamps, direction, session_id) to pair request-response\n        messages and check that clusters correspond across client/server\n        sides.\n\n        If interaction_metadata is None or lacks session/pairing info,\n        this constraint cannot be computed. A neutral score (0.5) is\n        returned with a warning.\n        """\n        if len(X) != len(labels):\n            raise ValueError("labels and X must contain the same number of messages")\n        if interaction_metadata is None:\n            warnings.warn(\n                "Remote coupling constraint requires interaction_metadata "\n                "(source/dest IP, ports, direction, session_id, timestamps). "\n                "Returning neutral score 0.5 — NOT paper-accurate.",\n                UserWarning,\n                stacklevel=2,\n            )\n            return 0.5\n\n        n = len(labels)\n        if len(interaction_metadata) != n:\n            raise ValueError("interaction_metadata must contain one entry per message")\n        if n < 2:\n            return 0.0\n\n        # Extract direction and session info from metadata.\n        directions: List[str] = []\n        session_ids: List[Any] = []\n        timestamps: List[float] = []\n\n        for meta in interaction_metadata:\n            directions.append(meta.get("direction", "unknown"))\n            session_ids.append(meta.get("session_id"))\n            timestamps.append(meta.get("timestamp", 0.0))\n\n        # Check if we have enough metadata for pairing.\n        has_sessions = any(s is not None for s in session_ids)\n        has_directions = any(d != "unknown" for d in directions)\n\n        if not has_directions:\n            warnings.warn(\n                "Remote coupling constraint requires \'direction\' in "\n                "interaction_metadata. Returning neutral 0.5.",\n                UserWarning,\n                stacklevel=2,\n            )\n            return 0.5\n\n        # Separate messages by direction.\n        client_indices = [i for i in range(n) if directions[i] == "client"]\n        server_indices = [i for i in range(n) if directions[i] == "server"]\n\n        if not client_indices or not server_indices:\n            # All messages from one direction — cannot check coupling.\n            return 0.5\n\n        # Build request-response pairs.\n        # Strategy: within each session, pair client messages with the\n        # nearest subsequent server message by timestamp.\n        pairs: List[Tuple[int, int]] = []\n\n        if has_sessions:\n            # Group by session_id.\n            client_by_session: Dict[Any, List[int]] = defaultdict(list)\n            server_by_session: Dict[Any, List[int]] = defaultdict(list)\n\n            for i in client_indices:\n                client_by_session[session_ids[i]].append(i)\n            for i in server_indices:\n                server_by_session[session_ids[i]].append(i)\n\n            for sess_id in client_by_session:\n                if sess_id not in server_by_session:\n                    continue\n                c_msgs = sorted(\n                    client_by_session[sess_id],\n                    key=lambda i: timestamps[i],\n                )\n                s_msgs = sorted(\n                    server_by_session[sess_id],\n                    key=lambda i: timestamps[i],\n                )\n\n                # Pair each request with the nearest unused subsequent response.\n                available_servers = list(s_msgs)\n                for ci in c_msgs:\n                    matching = [si for si in available_servers if timestamps[si] >= timestamps[ci]]\n                    if matching:\n                        best_si = min(matching, key=lambda si: timestamps[si] - timestamps[ci])\n                        pairs.append((ci, best_si))\n                        available_servers.remove(best_si)\n        else:\n            # No session info — pair by timestamp proximity globally.\n            sorted_clients = sorted(client_indices, key=lambda i: timestamps[i])\n            sorted_servers = sorted(server_indices, key=lambda i: timestamps[i])\n\n            available_servers = list(sorted_servers)\n            for ci in sorted_clients:\n                matching = [si for si in available_servers if timestamps[si] >= timestamps[ci]]\n                if matching:\n                    best_si = min(matching, key=lambda si: timestamps[si] - timestamps[ci])\n                    pairs.append((ci, best_si))\n                    available_servers.remove(best_si)\n\n        if not pairs:\n            return 0.5\n\n        # Check cluster correspondence: for each client cluster,\n        # what fraction of its paired server messages land in a\n        # single dominant server cluster?\n        # Build cluster -> paired cluster mapping.\n        client_cluster_pairs: Dict[int, List[int]] = defaultdict(list)\n\n        for ci, si in pairs:\n            client_cluster_pairs[int(labels[ci])].append(int(labels[si]))\n\n        cluster_correspondence_scores: List[float] = []\n\n        for c_cluster, s_clusters in client_cluster_pairs.items():\n            if not s_clusters:\n                continue\n            # Find dominant server cluster.\n            s_counts: Dict[int, int] = defaultdict(int)\n            for sc in s_clusters:\n                s_counts[sc] += 1\n            dominant_ratio = max(s_counts.values()) / len(s_clusters)\n            cluster_correspondence_scores.append(dominant_ratio)\n\n        if not cluster_correspondence_scores:\n            return 0.5\n\n        return float(np.mean(cluster_correspondence_scores))\n\n    # ==============================================================\n    #  3. Structural Consistency Constraint\n    # ==============================================================\n\n    @staticmethod\n    def structural_consistency(\n        labels: np.ndarray,\n        candidate: Dict[str, Any],\n        X: Optional[List[bytes]] = None,\n    ) -> float:\n        """\n        Netplier Structure Coherence Constraint.\n\n        Netplier: "messages of the same type share similar field\n        structure." After clustering by candidate field, re-aligns\n        messages within each cluster and computes alignment gap ratio.\n        p_s = 1 - (avg_gaps / total_length).\n\n        RPKClust adaptation: Since RPKClust does not use MSA, we adapt\n        this constraint to check full-message structural consistency\n        within each cluster using the message data X.\n\n        Requires X (full message set) to avoid circularity — clustering\n        by candidate values and then checking candidate value consistency\n        would be tautological. We check whether full messages within\n        each cluster share similar structure (byte-level consistency\n        at non-keyword offsets, length consistency, TLV validity).\n\n        Falls back to candidate-only checks if X is not provided\n        (labeled as NOT paper-accurate).\n        """\n        values = candidate["values"]\n        cand_type = candidate.get("type", "FOR")\n        if len(labels) != len(values):\n            raise ValueError("labels and candidate values must contain the same number of messages")\n        if X is not None and len(X) != len(values):\n            raise ValueError("X and candidate values must contain the same number of messages")\n\n        n_total = len(values)\n        if n_total == 0:\n            return 0.0\n\n        valid_mask = [v is not None for v in values]\n        presence = sum(valid_mask) / n_total\n\n        if presence < 0.5:\n            return 0.1\n\n        # Build cluster -> indices mapping.\n        clusters: Dict[int, List[int]] = {}\n        for idx, label in enumerate(labels):\n            clusters.setdefault(int(label), []).append(idx)\n\n        if X is not None:\n            # ---- Full-message structural consistency (preferred) ----\n            # Netplier: messages in the same cluster should share\n            # similar field structure. Without MSA, we measure this as\n            # intra-cluster message length consistency and byte-level\n            # agreement at non-keyword offsets.\n            cluster_scores: List[float] = []\n\n            for cluster_id, indices in clusters.items():\n                if len(indices) < 2:\n                    continue\n\n                cluster_msgs = [X[i] for i in indices]\n\n                # 1. Length consistency: messages in the same cluster\n                #    should have similar lengths (low variance).\n                lengths = [len(m) for m in cluster_msgs]\n                mean_len = np.mean(lengths)\n                if mean_len > 0:\n                    len_cv = np.std(lengths) / mean_len  # coefficient of variation\n                    len_score = 1.0 / (1.0 + len_cv)\n                else:\n                    len_score = 1.0\n\n                # 2. Byte-level structural agreement at non-keyword\n                #    positions: for each offset not covered by the\n                #    candidate field, check how consistent the bytes\n                #    are across messages in the cluster.\n                cand_offset = candidate.get("offset", 0)\n                cand_width = candidate.get("width", 1)\n\n                # Build set of offsets occupied by the candidate field.\n                cand_offsets = set()\n                if cand_type == "FOR":\n                    cand_offsets = set(range(cand_offset, cand_offset + cand_width))\n                # For NFOR, candidate offsets vary per message, so\n                # we skip offset-level exclusion and just use length.\n\n                min_len = min(lengths)\n                if min_len > 0:\n                    agreement_scores = []\n                    for pos in range(min_len):\n                        if pos in cand_offsets:\n                            continue  # Skip keyword field positions\n                        col_vals = [m[pos] for m in cluster_msgs]\n                        unique_count = len(set(col_vals))\n                        if unique_count == 1:\n                            agreement_scores.append(1.0)\n                        else:\n                            agreement_scores.append(1.0 / unique_count)\n\n                    if agreement_scores:\n                        byte_agreement = float(np.mean(agreement_scores))\n                    else:\n                        byte_agreement = 1.0\n                else:\n                    byte_agreement = 1.0\n\n                # Combine length consistency and byte agreement.\n                consistency = 0.4 * len_score + 0.6 * byte_agreement\n                cluster_scores.append(consistency)\n\n            if not cluster_scores:\n                return float(presence * 0.5)\n\n            avg_consistency = float(np.mean(cluster_scores))\n            p_s = 0.2 * presence + 0.8 * avg_consistency\n\n        else:\n            # ---- Fallback: candidate-only checks (NOT paper-accurate) ----\n            warnings.warn(\n                "structural_consistency called without X (full messages). "\n                "Using candidate-only fallback — NOT paper-accurate due to "\n                "circularity (clustering by candidate values then checking "\n                "candidate value consistency).",\n                UserWarning,\n                stacklevel=2,\n            )\n\n            cluster_scores: List[float] = []\n\n            for cluster_id, indices in clusters.items():\n                cluster_vals = [\n                    values[i] for i in indices\n                    if values[i] is not None\n                ]\n\n                if len(cluster_vals) < 2:\n                    continue\n\n                if cand_type == "NFOR":\n                    # Check TLV structural validity.\n                    patterns = candidate.get("patterns", [])\n                    valid_count = 0\n                    for p in patterns:\n                        if not isinstance(p, dict):\n                            continue\n                        len_val = p.get("len_val", -1)\n                        value_bytes = p.get("value_bytes", b"")\n                        if len_val == len(value_bytes):\n                            valid_count += 1\n                    consistency = valid_count / len(patterns) if patterns else 0.5\n                else:\n                    # FOR: use byte-level uniqueness as a weak proxy.\n                    unique_vals = set(\n                        bytes(v) if isinstance(v, (bytes, bytearray))\n                        else v\n                        for v in cluster_vals\n                    )\n                    uniqueness_ratio = len(unique_vals) / len(cluster_vals)\n                    consistency = 1.0 - uniqueness_ratio * 0.5\n\n                cluster_scores.append(consistency)\n\n            if not cluster_scores:\n                return float(presence * 0.5)\n\n            avg_consistency = float(np.mean(cluster_scores))\n            p_s = 0.3 * presence + 0.7 * avg_consistency\n\n        return float(np.clip(p_s, 0.0, 1.0))\n\n    # ==============================================================\n    #  4. Dimensional Constraint\n    # ==============================================================\n\n    @staticmethod\n    def dimensional_constraint(labels: np.ndarray) -> float:\n        """\n        Netplier Dimension Constraint.\n\n        Netplier considers two metrics:\n        1. r_distinct_value = (number of distinct field values) /\n           (number of messages) — compared to threshold t_value = 0.5.\n           If r > t_value, too many clusters → unlikely keyword.\n\n        2. r_single = (number of single-message clusters) /\n           (number of clusters) — compared to threshold t_single = 0.5.\n           If r > t_single, too many singleton clusters → unlikely keyword.\n\n        If both metrics are below their thresholds, p_d = 0.95 (high).\n        Otherwise, p_d = 0.1 (low).\n\n        The thresholds are conservatively set at 0.5 to avoid\n        discarding true keywords.\n        """\n        n = len(labels)\n        if n == 0:\n            return 0.0\n\n        k = len(np.unique(labels))\n\n        if k <= 1:\n            # Single cluster — degenerate case.\n            return 0.1\n\n        # Metric 1: distinct value ratio.\n        r_distinct_value = k / n\n\n        # Metric 2: single-message cluster ratio.\n        cluster_sizes: Dict[int, int] = {}\n        for label in labels:\n            cluster_sizes[int(label)] = cluster_sizes.get(int(label), 0) + 1\n\n        single_message_clusters = sum(\n            1 for size in cluster_sizes.values() if size == 1\n        )\n\n        # Netplier thresholds (conservatively set at 0.5).\n        # Paper: "If both values are less than their thresholds, the\n        # probability of the dimension constraint is high, e.g., 0.95.\n        # Otherwise, it is set to a low probability, e.g., 0.1."\n        # Use <= to be inclusive at the boundary (conservative).\n        t_value = 0.5\n        t_single = 0.5\n        r_single = single_message_clusters / k if k > 0 else 1.0\n\n        if r_distinct_value <= t_value and r_single <= t_single:\n            return 0.95\n        else:\n            return 0.1', 'rpkclust/metrics.py': '"""\nRPKClust Evaluation Metrics (Section 4).\n\nPaper Section 4.2 uses three primary metrics:\n    - Homogeneity (Eq. 18): 1 - H(T|C) / H(T)\n    - Completeness (Eq. 19): 1 - H(C|T) / H(C)\n    - V-Measure (Eq. 20): 2 * h * c / (h + c)\n\nPaper Section 4.5 also reports:\n    - Execution time (seconds)\n    - Memory overhead (MB)\n\nPaper Section 4.3 reports:\n    - FOR-NFOR boundary inference error (offset difference)\n\nSupplementary metrics (NOT in paper, useful for analysis):\n    - ARI, NMI, Silhouette, Davies-Bouldin, Clustering Accuracy\n"""\n\nimport time\nimport numpy as np\nfrom typing import Dict, Any, Optional, List, Tuple\nfrom sklearn.metrics import (\n    homogeneity_score,\n    completeness_score,\n    v_measure_score,\n    silhouette_score,\n    adjusted_rand_score,\n    normalized_mutual_info_score,\n    davies_bouldin_score,\n)\nfrom scipy.optimize import linear_sum_assignment\n\n\ndef convert_bytes_to_feature_matrix(\n    messages: List[bytes],\n    max_len: Optional[int] = None,\n) -> np.ndarray:\n    """\n    Converts a list of byte strings into a fixed-width numerical feature\n    matrix suitable for internal clustering metrics (Silhouette, DB).\n    Pads shorter messages with 0s and crops longer messages.\n    """\n    if max_len is None:\n        max_len = max(len(m) for m in messages) if messages else 0\n\n    if max_len == 0:\n        return np.zeros((len(messages), 1), dtype=np.float64)\n\n    matrix = np.zeros((len(messages), max_len), dtype=np.float64)\n    for i, m in enumerate(messages):\n        length = min(len(m), max_len)\n        matrix[i, :length] = np.frombuffer(m[:length], dtype=np.uint8)\n    return matrix\n\n\ndef clustering_accuracy(labels_true: np.ndarray, labels_pred: np.ndarray) -> float:\n    """\n    Clustering accuracy with optimal label matching using the Hungarian\n    algorithm. Cluster IDs are arbitrary, so we find the optimal assignment\n    between predicted and true labels that maximizes accuracy.\n\n    Returns a float in [0, 1].\n    """\n    labels_true = np.asarray(labels_true)\n    labels_pred = np.asarray(labels_pred)\n\n    if labels_true.ndim != 1 or labels_pred.ndim != 1:\n        raise ValueError("labels_true and labels_pred must be one-dimensional")\n    if len(labels_true) != len(labels_pred):\n        raise ValueError("labels_true and labels_pred must have the same length")\n    if len(labels_true) == 0:\n        return 0.0\n\n    # Build contingency matrix.\n    true_classes = np.unique(labels_true)\n    pred_classes = np.unique(labels_pred)\n\n    # Map labels to indices.\n    true_map = {label: i for i, label in enumerate(true_classes)}\n    pred_map = {label: i for i, label in enumerate(pred_classes)}\n\n    n_true = len(true_classes)\n    n_pred = len(pred_classes)\n\n    contingency = np.zeros((n_true, n_pred), dtype=int)\n    for t, p in zip(labels_true, labels_pred):\n        contingency[true_map[t], pred_map[p]] += 1\n\n    # Hungarian algorithm to find optimal assignment.\n    # linear_sum_assignment minimizes cost, so negate.\n    row_ind, col_ind = linear_sum_assignment(-contingency)\n\n    # Accuracy = correctly assigned / total.\n    correct = contingency[row_ind, col_ind].sum()\n    accuracy = correct / len(labels_true)\n\n    return float(accuracy)\n\n\ndef measure_memory_usage() -> float:\n    """\n    Returns current memory usage in MB.\n    Uses psutil if available, otherwise returns 0.0.\n    """\n    try:\n        import psutil\n        import os\n        process = psutil.Process(os.getpid())\n        return round(process.memory_info().rss / (1024 * 1024), 2)\n    except ImportError:\n        return 0.0\n\n\ndef evaluate_boundary(\n    true_boundary: int,\n    inferred_boundary: int,\n) -> Dict[str, Any]:\n    """\n    Paper Section 4.3: FOR-NFOR boundary inference evaluation.\n\n    Returns dict with:\n        - true_offset: ground truth boundary\n        - inferred_offset: inferred boundary\n        - error: difference (inferred - true)\n        - error_percentage: |error| / true_offset * 100\n    """\n    error = inferred_boundary - true_boundary\n    error_pct = abs(error) / true_boundary * 100 if true_boundary > 0 else 0.0\n\n    return {\n        "True Offset": true_boundary,\n        "Inferred Offset": inferred_boundary,\n        "Error": error,\n        "Error (%)": round(error_pct, 2),\n    }\n\n\ndef evaluate_clustering(\n    labels_true: np.ndarray,\n    labels_pred: np.ndarray,\n    feature_matrix: Optional[np.ndarray] = None,\n    exec_time: float = 0.0,\n    memory_mb: Optional[float] = None,\n) -> Dict[str, Any]:\n    """\n    Evaluates clustering performance using paper metrics (Section 4.2)\n    and supplementary analysis metrics.\n\n    Paper metrics (primary):\n        - Homogeneity (Eq. 18)\n        - Completeness (Eq. 19)\n        - V-Measure (Eq. 20)\n        - Execution Time\n        - Memory Overhead\n\n    Supplementary metrics (NOT in paper):\n        - ARI, NMI, Clustering Accuracy\n        - Silhouette, Davies-Bouldin (require feature_matrix)\n\n    Parameters\n    ----------\n    labels_true : array-like\n        Ground truth message type labels.\n    labels_pred : array-like\n        Predicted cluster labels.\n    feature_matrix : optional np.ndarray\n        Numeric feature matrix for internal clustering metrics.\n    exec_time : float\n        Execution time in seconds.\n    memory_mb : optional float\n        Memory usage in MB. If None, measured automatically.\n    """\n    labels_true = np.asarray(labels_true)\n    labels_pred = np.asarray(labels_pred)\n    if labels_true.ndim != 1 or labels_pred.ndim != 1:\n        raise ValueError("labels_true and labels_pred must be one-dimensional")\n    if len(labels_true) != len(labels_pred):\n        raise ValueError("labels_true and labels_pred must have the same length")\n    if len(labels_true) == 0:\n        raise ValueError("at least one label is required for clustering evaluation")\n    if feature_matrix is not None and np.asarray(feature_matrix).shape[0] != len(labels_true):\n        raise ValueError("feature_matrix must contain one row per label")\n\n    if memory_mb is None:\n        memory_mb = measure_memory_usage()\n\n    # Mask for internal metrics (exclude noise points).\n    valid_mask = labels_pred != -1\n    num_valid = int(np.sum(valid_mask))\n    num_clusters = (\n        len(set(labels_pred[valid_mask])) if num_valid > 0 else 0\n    )\n\n    # ---- Paper Metrics (Primary) ----\n    metrics: Dict[str, Any] = {\n        # Paper Eq. 18: Homogeneity\n        "Homogeneity": round(\n            homogeneity_score(labels_true, labels_pred), 4\n        ),\n        # Paper Eq. 19: Completeness\n        "Completeness": round(\n            completeness_score(labels_true, labels_pred), 4\n        ),\n        # Paper Eq. 20: V-Measure\n        "V-Measure": round(\n            v_measure_score(labels_true, labels_pred), 4\n        ),\n        # Paper Section 4.5: Execution Time\n        "Execution Time (s)": round(exec_time, 4),\n        # Paper Section 4.5: Memory Overhead\n        "Memory (MB)": memory_mb,\n        # Paper Section 4.2: Clusters Found\n        "Clusters Found": num_clusters,\n    }\n\n    # ---- Supplementary Metrics (NOT in paper) ----\n    metrics["ARI"] = round(\n        adjusted_rand_score(labels_true, labels_pred), 4\n    )\n    metrics["NMI"] = round(\n        normalized_mutual_info_score(labels_true, labels_pred), 4\n    )\n    metrics["Clustering Accuracy"] = round(\n        clustering_accuracy(labels_true, labels_pred), 4\n    )\n\n    # Internal metrics require feature matrix and >= 2 clusters.\n    if (\n        feature_matrix is not None\n        and num_clusters >= 2\n        and num_valid > num_clusters\n    ):\n        try:\n            metrics["Silhouette"] = round(\n                silhouette_score(\n                    feature_matrix[valid_mask],\n                    labels_pred[valid_mask],\n                ),\n                4,\n            )\n            metrics["Davies-Bouldin"] = round(\n                davies_bouldin_score(\n                    feature_matrix[valid_mask],\n                    labels_pred[valid_mask],\n                ),\n                4,\n            )\n        except ValueError:\n            metrics["Silhouette"] = np.nan\n            metrics["Davies-Bouldin"] = np.nan\n    else:\n        metrics["Silhouette"] = np.nan\n        metrics["Davies-Bouldin"] = np.nan\n\n    return metrics\n\n\ndef evaluate_keyword_inference(\n    candidates: List[Dict[str, Any]],\n    true_keyword_offset: Any,\n) -> Dict[str, Any]:\n    """\n    Paper Section 4.4: Keyword inference evaluation.\n\n    Evaluates whether the real keyword field was correctly identified\n    and ranked first.\n\n    Parameters\n    ----------\n    candidates : list of candidate dicts\n        Must be sorted by \'prob\' descending (as output by RPKClust pipeline).\n    true_keyword_offset : int or tuple\n        The byte offset(s) of the real keyword field(s).\n    """\n    if not candidates:\n        return {\n            "True Keyword Offset": true_keyword_offset,\n            "Inferred Keyword Offset": None,\n            "Correct": False,\n            "Rank": None,\n            "Probability": None,\n        }\n\n    best = candidates[0]\n    inferred_offset = best.get("offset")\n    inferred_width = best.get("width", 1)\n    inferred_end = inferred_offset + inferred_width - 1 if inferred_offset is not None else None\n\n    # Handle various true_keyword_offset formats.\n    if isinstance(true_keyword_offset, (tuple, list)):\n        correct = inferred_offset in true_keyword_offset\n    elif isinstance(true_keyword_offset, str) and ":" in true_keyword_offset:\n        # e.g. "(16:18)" format.\n        parts = true_keyword_offset.strip("()").split(":")\n        t_start, t_end = int(parts[0]), int(parts[1])\n        correct = (\n            inferred_offset is not None\n            and inferred_end is not None\n            and inferred_offset >= t_start\n            and inferred_end <= t_end\n        )\n    else:\n        correct = inferred_offset == true_keyword_offset\n\n    # Find rank of true keyword.\n    # Compare spans (offset, offset+width-1) for multi-byte keywords.\n    rank = None\n    for i, cand in enumerate(candidates):\n        cand_start = cand.get("offset")\n        cand_end = cand_start + cand.get("width", 1) - 1 if cand_start is not None else None\n        if isinstance(true_keyword_offset, (tuple, list)):\n            # true_keyword_offset is a list/tuple of acceptable offsets.\n            if cand_start in true_keyword_offset:\n                rank = i + 1\n                break\n        elif isinstance(true_keyword_offset, str):\n            # e.g. "(16:18)" format — parse start:end.\n            if ":" in true_keyword_offset:\n                parts = true_keyword_offset.strip("()").split(":")\n                t_start, t_end = int(parts[0]), int(parts[1])\n                if cand_start is not None and cand_end is not None:\n                    if cand_start >= t_start and cand_end <= t_end:\n                        rank = i + 1\n                        break\n            else:\n                if cand_start == int(true_keyword_offset):\n                    rank = i + 1\n                    break\n        else:\n            if cand_start == true_keyword_offset:\n                rank = i + 1\n                break\n\n    return {\n        "True Keyword Offset": true_keyword_offset,\n        "Inferred Keyword Offset": inferred_offset,\n        "Correct": correct,\n        "Rank": rank,\n        "Probability": round(best.get("prob", 0.0), 4),\n        "Top-3 Candidates": [\n            {\n                "offset": c.get("offset"),\n                "prob": round(c.get("prob", 0.0), 4),\n                "type": c.get("type"),\n            }\n            for c in candidates[:3]\n        ],\n    }', 'rpkclust/optimizer.py': '"""\nRPKClust Section 3.6: Keyword Inference (keyword_inference.py)\n\nTwo-stage probability inference for protocol keyword identification.\n\nFirst Stage (Section 3.6, "The First Stage"):\n    - Cluster messages by each candidate\'s values\n    - Evaluate four Netplier constraints: message similarity, remote coupling,\n      structural consistency, dimensional\n    - Calculate posterior probability p_f for each candidate\n    - Rank candidates by p_f\n\nSecond Stage (Section 3.6, "The Second Stage"):\n    - Bit-use constraint p_bit (Eq. 7–10)\n    - Position constraint p_offset (Eq. 11)\n    - Final posterior P(K=1 | p_bit, p_offset) = M / (M + N) (Eq. 12–15)\n    - Select field with highest probability as the keyword\n\nPaper: "We take the probability of cluster constraint inference as the prior\nprobability, that is, P(K=1) = p_f and P(K=0) = 1 - p_f."\n"""\n\nimport numpy as np\nfrom typing import List, Dict, Any, Optional, Tuple\n\nfrom .constraints import ClusteringConstraints\n\n\n# ------------------------------------------------------------------\n#  Helpers\n# ------------------------------------------------------------------\n\n_MISSING = object()  # sentinel for absent NFOR TLV values\n\n\ndef _val_to_int(v) -> int:\n    """Safely converts candidate field values to integer representation."""\n    if isinstance(v, int):\n        return v\n    if isinstance(v, (bytes, bytearray)):\n        return int.from_bytes(v, \'big\')\n    if isinstance(v, tuple):\n        b_list = []\n        for item in v:\n            if isinstance(item, (bytes, bytearray)):\n                b_list.append(item)\n            elif isinstance(item, int):\n                b_list.append(bytes([item]))\n        return int.from_bytes(b\'\'.join(b_list), \'big\')\n    if isinstance(v, str):\n        return int.from_bytes(v.encode(\'utf-8\'), \'big\')\n    return int(v)\n\n\n# ------------------------------------------------------------------\n#  Two-Stage Bayesian Inference Model\n# ------------------------------------------------------------------\n\nclass RPKClustOptimizer:\n    """\n    Two-Stage Bayesian Inference Model for Protocol Keyword Identification.\n\n    Stage 1 produces p_f (prior probability from clustering constraints).\n    Stage 2 combines p_f with p_bit and p_offset to produce the final\n    posterior probability P(K=1 | p_bit, p_offset).\n    """\n\n    # ==============================================================\n    #  First Stage: Clustering Constraint Inference\n    # ==============================================================\n\n    def compute_stage1_probability(\n        self,\n        candidate: Dict[str, Any],\n        X: List[bytes],\n        interaction_metadata: Optional[List[Dict[str, Any]]] = None,\n        prior: float = 0.1,\n    ) -> Tuple[float, Dict[str, float]]:\n        """\n        First Stage: compute p_f (prior probability) from four Netplier\n        clustering constraints.\n\n        Paper: "We traverse this list and cluster the messages according to\n        the values of each candidate. Then we observe whether the clustering\n        results meet the clustering constraints."\n\n        Parameters\n        ----------\n        candidate : dict\n            Candidate field with \'values\' key (list of bytes/None per message).\n        X : List[bytes]\n            Full message set.\n        interaction_metadata : optional\n            Per-message metadata (source/dest IP, ports, timestamps) needed\n            for remote coupling constraint. If None, remote coupling may\n            be degraded.\n        prior : float\n            Prior probability P(K=1) for a random candidate being a keyword.\n            The paper does not specify this value for Stage 1. Default 0.1\n            reflects that keyword fields are rare among all candidates.\n\n        Returns\n        -------\n        p_f : float\n            Posterior probability from constraint inference (used as prior\n            in Stage 2).\n        constraint_values : dict\n            Individual constraint scores for debugging/ranking.\n        """\n        if X is None:\n            raise ValueError("X must be a sequence of messages, not None")\n        values = candidate.get("values")\n        if values is None or len(values) != len(X):\n            raise ValueError("candidate values must contain one entry per message")\n        if interaction_metadata is not None and len(interaction_metadata) != len(X):\n            raise ValueError("interaction_metadata must contain one entry per message")\n        if not np.isfinite(prior) or not 0.0 < prior < 1.0:\n            raise ValueError("prior must be a finite probability strictly between 0 and 1")\n        labels = self.cluster_by_candidate(values)\n\n        c1 = ClusteringConstraints.message_similarity(labels, X)\n        # remote_coupling may need interaction metadata; fall back to\n        # the original 2-arg signature for compatibility.\n        try:\n            c2 = ClusteringConstraints.remote_coupling(\n                labels, X, interaction_metadata\n            )\n        except TypeError:\n            c2 = ClusteringConstraints.remote_coupling(labels, X)\n        c3 = ClusteringConstraints.structural_consistency(\n            labels, candidate, X=X\n        )\n        c4 = ClusteringConstraints.dimensional_constraint(labels)\n\n        constraint_values = {\n            "message_similarity": float(c1),\n            "remote_coupling": float(c2),\n            "structural_consistency": float(c3),\n            "dimensional_constraint": float(c4),\n        }\n\n        p_f = self._constraint_bayesian_update(\n            c1, c2, c3, c4, prior=prior,\n        )\n\n        return p_f, constraint_values\n\n    def rank_stage1_candidates(\n        self,\n        candidates: List[Dict[str, Any]],\n        X: List[bytes],\n        interaction_metadata: Optional[List[Dict[str, Any]]] = None,\n        prior: float = 0.1,\n    ) -> List[Tuple[Dict[str, Any], float, Dict[str, float]]]:\n        """\n        First Stage ranking: traverse all candidates, compute p_f for each,\n        and rank by probability.\n\n        Paper: "Based on the probability inference of the above constraints,\n        we initially rank the keyword fields according to their probabilities."\n\n        Returns\n        -------\n        List of (candidate, p_f, constraint_values) sorted by p_f descending.\n        """\n        scored: List[Tuple[Dict[str, Any], float, Dict[str, float]]] = []\n\n        for candidate in candidates:\n            p_f, constraints = self.compute_stage1_probability(\n                candidate, X, interaction_metadata=interaction_metadata,\n                prior=prior,\n            )\n            scored.append((candidate, p_f, constraints))\n\n        scored.sort(key=lambda t: t[1], reverse=True)\n        return scored\n\n    def cluster_by_candidate(self, values: List[Any]) -> np.ndarray:\n        """\n        Cluster messages by candidate field values.\n        Messages with the same value are assigned the same cluster label.\n        Uses _MISSING sentinel for None values so absent NFOR TLVs\n        do not collide with real string values.\n        """\n        mapping: Dict[Any, int] = {}\n        labels: List[int] = []\n\n        for v in values:\n            if v is None:\n                key = _MISSING\n            elif isinstance(v, (bytes, bytearray)):\n                key = bytes(v)\n            else:\n                key = v\n\n            if key not in mapping:\n                mapping[key] = len(mapping)\n            labels.append(mapping[key])\n\n        return np.array(labels)\n\n    def _constraint_bayesian_update(\n        self,\n        c1: float,\n        c2: float,\n        c3: float,\n        c4: float,\n        prior: float = 0.1,\n    ) -> float:\n        """\n        Naive Bayes posterior for Stage 1 constraint inference.\n\n        NOTE: The paper does not specify the exact formula for Stage 1\n        probability. It says the inference is "similar to the factor graph\n        used in Netplier." This implementation uses a naive Bayes approach\n        (independent constraint factors), which is a reasonable approximation.\n\n        Paper: "We calculate the posterior probability of each field being\n        a keyword field."\n        """\n        if not np.isfinite(prior) or not 0.0 < prior < 1.0:\n            raise ValueError("prior must be a finite probability strictly between 0 and 1")\n        constraints = np.array([c1, c2, c3, c4], dtype=float)\n        constraints = np.clip(constraints, 1e-6, 1 - 1e-6)\n\n        # Likelihood P(D | K=1) = product of constraint satisfactions.\n        likelihood_keyword = float(np.prod(constraints))\n\n        # Likelihood P(D | K=0) = product of constraint violations.\n        likelihood_not_keyword = float(np.prod(1.0 - constraints))\n\n        numerator = likelihood_keyword * prior\n        denominator = (\n            numerator\n            + likelihood_not_keyword * (1.0 - prior)\n        )\n\n        if denominator <= 0:\n            return 1e-6\n\n        posterior = numerator / denominator\n        return float(np.clip(posterior, 1e-6, 1.0 - 1e-6))\n\n    # ==============================================================\n    #  Second Stage: Self-Constraint Inference\n    # ==============================================================\n\n    def compute_p_bit(self, values: List[Any]) -> float:\n        """\n        Bit-Use Constraint Probability p_bit (Eq. 7–10).\n\n        Paper:\n            MSB = highest bit position used by any value in the field.\n            Q(k) = proportion of values where MSB >= k.\n            P(k) = 1 - 1 / (2^(MSB+1-k)).\n            D = sqrt(sum((Q(k) - P(k))^2)).\n            Dmax = max over concentration points m of:\n                sqrt(sum_{k=0}^{m}(1-P(k))^2 + sum_{k=m+1}^{MSB}(P(k))^2).\n            p_bit = 1 - D / Dmax.\n        """\n        valid_vals = [_val_to_int(v) for v in values if v is not None]\n        if not valid_vals:\n            return 1e-6\n\n        # Constant field filter: single unique value carries no keyword info.\n        if len(set(valid_vals)) <= 1:\n            return 1e-6\n\n        # Paper: "traverse all values and determine the position of the\n        # Most Significant Bit (MSB)."\n        # MSB = highest bit position used by any value.\n        max_val = max(valid_vals)\n        if max_val == 0:\n            return 1e-6\n\n        msb = max_val.bit_length() - 1  # 0-based MSB position\n\n        # Paper: "Q(k) = proportion of values where MSB >= k"\n        # For each value, compute its individual MSB, then count how many\n        # values have individual MSB >= k.\n        individual_msbs = []\n        for val in valid_vals:\n            if val == 0:\n                individual_msbs.append(0)\n            else:\n                individual_msbs.append(val.bit_length() - 1)\n\n        n = len(valid_vals)\n        q_k = np.zeros(msb + 1)\n        for k in range(msb + 1):\n            count = sum(1 for ind_msb in individual_msbs if ind_msb >= k)\n            q_k[k] = count / n\n\n        # Paper Eq. (7): P(k) = 1 - 1 / (2^(MSB+1-k))\n        p_k = np.zeros(msb + 1)\n        for k in range(msb + 1):\n            p_k[k] = 1.0 - 1.0 / (2.0 ** (msb + 1 - k))\n\n        # Paper Eq. (8): D = sqrt(sum((Q(k) - P(k))^2))\n        euclidean_dist = float(np.sqrt(np.sum((q_k - p_k) ** 2)))\n\n        # Paper Eq. (9): Dmax = max over concentration points m\n        # Dmax = sqrt(sum_{k=0}^{m}(1-P(k))^2 + sum_{k=m+1}^{MSB}(P(k))^2)\n        # The concentration point m represents the extreme case where all\n        # values concentrate at a single bit position.\n        d_max = 0.0\n        for m in range(msb + 1):\n            # Q concentrated at bit m: Q(k)=1 for k<=m, Q(k)=0 for k>m\n            sum_low = sum((1.0 - p_k[k]) ** 2 for k in range(0, m + 1))\n            sum_high = sum((p_k[k]) ** 2 for k in range(m + 1, msb + 1))\n            d_candidate = np.sqrt(sum_low + sum_high)\n            if d_candidate > d_max:\n                d_max = d_candidate\n\n        # Paper Eq. (10): p_bit = 1 - D / Dmax\n        if d_max <= 0:\n            return 1e-6\n\n        p_bit = 1.0 - euclidean_dist / d_max\n        return float(np.clip(p_bit, 1e-6, 1.0 - 1e-6))\n\n    def compute_p_offset(\n        self,\n        candidate_type: str,\n        offset: int = 0,\n        boundary_B: int = 0,\n    ) -> float:\n        """\n        Position Constraint Probability p_offset (Eq. 11).\n\n        Paper:\n            p_offset = max(0.95 - 0.01 * cand_offset, 0.7)  if cand in FOR\n            p_offset = 0.60                                  if cand in NFOR\n\n        Parameters\n        ----------\n        candidate_type : str\n            "FOR" or "NFOR".\n        offset : int\n            For FOR candidates: the byte offset within the message.\n            For NFOR candidates: not used (fixed 0.60).\n        boundary_B : int\n            Not used directly; included for interface consistency.\n        """\n        if candidate_type == "FOR":\n            # Paper Eq. (11): max(0.95 - 0.01 * cand_offset, 0.7)\n            return max(0.95 - 0.01 * offset, 0.7)\n        if candidate_type == "NFOR":\n            # Paper Eq. (11): 0.60 for NFOR\n            return 0.60\n        raise ValueError("candidate_type must be \'FOR\' or \'NFOR\'")\n\n    def bayesian_update(self, p_bit: float, p_offset: float, p_f: float) -> float:\n        """\n        Final posterior probability P(K=1 | p_bit, p_offset) (Eq. 12–15).\n\n        Paper:\n            f_bit(K=1) = p_bit,     f_bit(K=0) = 1 - p_bit\n            f_offset(K=1) = p_offset, f_offset(K=0) = 1 - p_offset\n            P(K, p_bit, p_offset) ∝ f_bit(K) * f_offset(K) * P(K)\n\n            M = p_bit * p_offset * p_f\n            N = (1 - p_bit) * (1 - p_offset) * (1 - p_f)\n            P(K=1 | p_bit, p_offset) = M / (M + N)\n        """\n        M = p_bit * p_offset * p_f\n        N = (1.0 - p_bit) * (1.0 - p_offset) * (1.0 - p_f)\n\n        denominator = M + N\n        if denominator <= 0:\n            return 1e-6\n\n        posterior = M / denominator\n        return float(np.clip(posterior, 1e-6, 1.0 - 1e-6))\n\n    # ==============================================================\n    #  Full Two-Stage Pipeline\n    # ==============================================================\n\n    def infer_keyword(\n        self,\n        candidates: List[Dict[str, Any]],\n        X: List[bytes],\n        interaction_metadata: Optional[List[Dict[str, Any]]] = None,\n        stage1_prior: float = 0.1,\n        top_k: Optional[int] = None,\n    ) -> Tuple[Optional[Dict[str, Any]], float, List[Tuple[Dict[str, Any], float]]]:\n        """\n        Full two-stage keyword inference pipeline.\n\n        Paper: "Select the field with the highest probability result as\n        the keyword field."\n\n        Parameters\n        ----------\n        candidates : List[dict]\n            Merged FOR + NFOR candidate list.\n        X : List[bytes]\n            Full message set.\n        interaction_metadata : optional\n            Per-message metadata for remote coupling constraint.\n        stage1_prior : float\n            Prior for Stage 1 naive Bayes.\n        top_k : optional int\n            If specified, only the top-k Stage 1 candidates proceed to\n            Stage 2. Paper says "fields ranked high in the results of the\n            first stage inference."\n\n        Returns\n        -------\n        best_candidate : dict or None\n            The candidate with highest final probability.\n        best_probability : float\n            Final posterior probability.\n        all_scored : list of (candidate, probability)\n            All candidates scored in Stage 2, sorted by probability.\n        """\n        if top_k is not None and (not isinstance(top_k, int) or isinstance(top_k, bool) or top_k <= 0):\n            raise ValueError("top_k must be a positive integer or None")\n\n        # --- Stage 1: ranking ---\n        stage1_ranked = self.rank_stage1_candidates(\n            candidates, X,\n            interaction_metadata=interaction_metadata,\n            prior=stage1_prior,\n        )\n\n        # Paper: "we conduct a new round of inference on the fields ranked\n        # high in the results of the first stage inference"\n        if top_k is not None:\n            stage1_ranked = stage1_ranked[:top_k]\n\n        # --- Stage 2: self-constraint inference ---\n        all_scored: List[Tuple[Dict[str, Any], float]] = []\n\n        for candidate, p_f, _ in stage1_ranked:\n            values = candidate["values"]\n\n            p_bit = self.compute_p_bit(values)\n\n            cand_type = candidate.get("type", "FOR")\n            cand_offset = candidate.get("offset", 0)\n            boundary_B = candidate.get("boundary_B", 0)\n            p_offset = self.compute_p_offset(cand_type, cand_offset, boundary_B)\n\n            # Paper Eq. (15): P(K=1 | p_bit, p_offset) = M / (M + N)\n            final_prob = self.bayesian_update(p_bit, p_offset, p_f)\n\n            all_scored.append((candidate, final_prob))\n\n        all_scored.sort(key=lambda t: t[1], reverse=True)\n\n        if not all_scored:\n            return None, 0.0, []\n\n        best_candidate, best_prob = all_scored[0]\n        return best_candidate, best_prob, all_scored', 'rpkclust/rpkclust.py': '"""\nRPKClust Protocol Field Discovery and Message Clustering Pipeline.\n\nPaper Sections 3.1–3.6:\n    1. FOR-NFOR Boundary Identification (Algorithm 1)\n    2. Region-Partitioned Candidate Generation (Algorithms 2 & 3)\n    3. Two-Stage Bayesian Inference (Section 3.6)\n    4. Semantic Clustering Assignment\n\nIntegration notes:\n    - Uses SemanticRules.identify_boundary + collect_semantic_regions\n      (shared _scan_semantic_regions to avoid drift).\n    - Passes semantic_regions to extract_for_candidates (Algorithm 2).\n    - Uses the revised RPKClustOptimizer with paper-accurate Eq. 7–15.\n    - Stage 1 ranks candidates by p_f before Stage 2 (paper: "fields\n      ranked high in the results of the first stage inference").\n"""\n\nimport numpy as np\nfrom typing import List, Dict, Any, Optional, Callable\n\nfrom .optimizer import RPKClustOptimizer\nfrom .semantic_rules import SemanticRules\nfrom .utils import extract_for_candidates, extract_nfor_tlv_candidates\n\n\nclass RPKClust:\n    """\n    RPKClust Protocol Field Discovery and Message Clustering Pipeline.\n    """\n\n    def __init__(self):\n        self.optimizer = RPKClustOptimizer()\n        self.boundary_B = 0\n        self.semantic_regions: List[Dict[str, Any]] = []\n        self.candidates: List[Dict[str, Any]] = []\n        self.best_candidate: Optional[Dict[str, Any]] = None\n        self.labels_: Optional[np.ndarray] = None\n\n    def fit(\n        self,\n        X: List[bytes],\n        interaction_metadata: Optional[List[Dict[str, Any]]] = None,\n        capture_start: Optional[float] = None,\n        capture_end: Optional[float] = None,\n        timezone_offset: float = 0.0,\n        direction_labels: Optional[List[Any]] = None,\n        t_len: int = 1,\n        l_len: int = 1,\n        validate_tlv: Optional[Callable[..., bool]] = None,\n        stage1_prior: float = 0.1,\n        top_k: Optional[int] = None,\n    ) -> "RPKClust":\n        """\n        Executes the full RPKClust pipeline on binary message trace X.\n\n        Parameters\n        ----------\n        X : List[bytes]\n            Application-layer binary messages.\n        interaction_metadata : optional\n            Per-message metadata (source/dest IP, ports, timestamps) for\n            the remote coupling constraint in Stage 1.\n        capture_start, capture_end : optional float\n            Capture time window for timestamp rule in boundary detection.\n        timezone_offset : float\n            Timezone offset in seconds for timestamp rule.\n        direction_labels : optional\n            Per-message direction labels for address rule.\n        t_len, l_len : int\n            TLV type and length field widths for NFOR candidate generation.\n        validate_tlv : optional callable\n            Semantic TLV validator for NFOR candidate generation.\n        stage1_prior : float\n            Prior probability for Stage 1 naive Bayes.\n        top_k : optional int\n            If set, only the top-k Stage 1 candidates proceed to Stage 2.\n            Paper: "we conduct a new round of inference on the fields ranked\n            high in the results of the first stage inference."\n        """\n        if X is None:\n            raise ValueError("X must be a sequence of messages, not None")\n        if not X:\n            self.boundary_B = 0\n            self.semantic_regions = []\n            self.candidates = []\n            self.best_candidate = None\n            self.labels_ = np.array([], dtype=int)\n            return self\n        if not 0.0 < float(stage1_prior) < 1.0:\n            raise ValueError("stage1_prior must be a finite probability strictly between 0 and 1")\n        if top_k is not None and (not isinstance(top_k, int) or isinstance(top_k, bool) or top_k <= 0):\n            raise ValueError("top_k must be a positive integer or None")\n        if interaction_metadata is not None and len(interaction_metadata) != len(X):\n            raise ValueError("interaction_metadata must contain one entry per message")\n        if direction_labels is not None and len(direction_labels) != len(X):\n            raise ValueError("direction_labels must contain one entry per message")\n\n        # ---- Step 1: Boundary Identification + Semantic Region Collection\n        print("Identifying Boundaries...")\n        self.boundary_B, _ = SemanticRules.identify_boundary(\n            X,\n            capture_start=capture_start,\n            capture_end=capture_end,\n            timezone_offset=timezone_offset,\n            direction_labels=direction_labels,\n        )\n        self.semantic_regions = SemanticRules.collect_semantic_regions(\n            X,\n            capture_start=capture_start,\n            capture_end=capture_end,\n            timezone_offset=timezone_offset,\n            direction_labels=direction_labels,\n        )\n        print(f"  Boundary B = {self.boundary_B}")\n        print(f"  Semantic regions: {len(self.semantic_regions)}")\n\n        # ---- Step 2: Region-Partitioned Candidate Generation\n        print("Extracting FOR Candidates...")\n        for_cands = extract_for_candidates(\n            X,\n            self.boundary_B,\n            semantic_regions=self.semantic_regions,\n        )\n        print(f"  FOR candidates: {len(for_cands)}")\n\n        print("Extracting NFOR Candidates...")\n        nfor_cands = extract_nfor_tlv_candidates(\n            X,\n            self.boundary_B,\n            t_len=t_len,\n            l_len=l_len,\n            validate_tlv=validate_tlv,\n        )\n        print(f"  NFOR candidates: {len(nfor_cands)}")\n\n        self.candidates = for_cands + nfor_cands\n        print(f"  Total candidates: {len(self.candidates)}")\n\n        # ---- Step 3: Two-Stage Bayesian Inference\n        print("Two Stage Bayesian Inference...")\n\n        # Stage 1: Rank all candidates by p_f (clustering constraints).\n        stage1_ranked = self.optimizer.rank_stage1_candidates(\n            self.candidates,\n            X,\n            interaction_metadata=interaction_metadata,\n            prior=stage1_prior,\n        )\n\n        # Paper: "fields ranked high in the results of the first stage\n        # inference" proceed to Stage 2.\n        if top_k is not None:\n            stage1_ranked = stage1_ranked[:top_k]\n\n        # Stage 2: Self-constraint inference (p_bit, p_offset, final posterior).\n        for candidate, p_f, constraint_values in stage1_ranked:\n            candidate["stage1_prob"] = p_f\n            candidate["constraint_values"] = constraint_values\n\n            p_bit = self.optimizer.compute_p_bit(candidate["values"])\n            candidate["p_bit"] = p_bit\n\n            cand_type = candidate.get("type", "FOR")\n            cand_offset = candidate.get("offset", 0)\n            p_offset = self.optimizer.compute_p_offset(\n                cand_type, cand_offset, self.boundary_B\n            )\n            candidate["p_offset"] = p_offset\n\n            final_prob = self.optimizer.bayesian_update(\n                p_bit, p_offset, p_f\n            )\n            candidate["prob"] = final_prob\n\n        # Update self.candidates to reflect scored results.\n        scored_candidates = [c for c, _, _ in stage1_ranked]\n        self.candidates = scored_candidates\n\n        # ---- Step 4: Keyword Selection\n        print("Keyword Selection...")\n        if self.candidates:\n            self.candidates.sort(key=lambda c: c["prob"], reverse=True)\n            self.best_candidate = self.candidates[0]\n            print(f"  Best: {self.best_candidate.get(\'tag\', \'?\')} "\n                  f"prob={self.best_candidate[\'prob\']:.4f}")\n        else:\n            self.best_candidate = None\n\n        # ---- Step 5: Semantic Clustering Assignment\n        print("Assigning Semantic Clustering...")\n        self.labels_ = self._assign_clusters(X)\n        return self\n\n    def _assign_clusters(self, X: List[bytes]) -> np.ndarray:\n        """\n        Assigns cluster labels based on the selected keyword field\'s values.\n        Reuses optimizer.cluster_by_candidate for consistency with Stage 1.\n        """\n        if not self.best_candidate:\n            return np.zeros(len(X), dtype=int)\n\n        vals = self.best_candidate["values"]\n        return self.optimizer.cluster_by_candidate(vals)\n\n    def fit_predict(self, X: List[bytes], **fit_kwargs) -> np.ndarray:\n        """Fit and return cluster labels. Accepts same kwargs as fit()."""\n        self.fit(X, **fit_kwargs)\n        if self.labels_ is None:\n            raise RuntimeError("fit completed without producing cluster labels")\n        return self.labels_', 'rpkclust/semantic_rules.py': '\n# RPKClust Boundary Identification & Semantic Rules (semantic_rules.py)\n# Generic evaluation of structural semantics across packet byte traces.\n\n\nimport numpy as np\nimport math\nimport zlib\nimport struct\nfrom typing import List, Set, Tuple, Any, Optional, Dict, Callable\n\n\nclass SemanticRules:\n    """\n    Evaluates semantic rules for boundary detection and field profiling.\n    All rules follow the formal definitions in RPKClust Section 3.3.\n    """\n\n    # ------------------------------------------------------------------\n    #  Helpers\n    # ------------------------------------------------------------------\n\n    @staticmethod\n    def _to_int_list(fragments: List[Any], width: int) -> Optional[List[int]]:\n        """\n        Converts a list of byte/int fragments into a uniform list of integers.\n        Returns None if any fragment is invalid, None, or has an incorrect byte length.\n        Validates every value is in [0, 2^(8*width) - 1].\n        """\n        max_value = (1 << (8 * width)) - 1\n        nums = []\n        for f in fragments:\n            if f is None:\n                return None\n            if isinstance(f, (bytes, bytearray)):\n                if len(f) != width:\n                    return None\n                nums.append(int.from_bytes(f, \'big\'))\n            elif isinstance(f, int):\n                if f < 0 or f > max_value:          # range-check every element\n                    return None\n                nums.append(f)\n            else:\n                return None\n        return nums\n\n    @staticmethod\n    def _extract_fragments(X: List[bytes], offset: int, width: int) -> Optional[List[bytes]]:\n        """\n        Extracts a slice [offset : offset + width] from each message in cluster X.\n        Returns None if any message is too short for the slice.\n        """\n        fragments = []\n        for msg in X:\n            if offset + width > len(msg):\n                return None\n            fragments.append(msg[offset:offset + width])\n        return fragments\n\n    @staticmethod\n    def _calc_internet_checksum(data: bytes) -> int:\n        """RFC 1071 Internet Checksum (16-bit ones\' complement sum)."""\n        if len(data) % 2 == 1:\n            data += b\'\\x00\'\n        words = struct.unpack(f">{len(data) // 2}H", data)\n        checksum = sum(words)\n        while checksum >> 16:\n            checksum = (checksum & 0xFFFF) + (checksum >> 16)\n        return (~checksum) & 0xFFFF\n\n    @staticmethod\n    def _calc_crc16_ccitt(data: bytes, poly: int = 0x1021, init: int = 0xFFFF) -> int:\n        """CRC-16/CCITT-FALSE checksum."""\n        crc = init\n        for byte in data:\n            crc ^= (byte << 8)\n            for _ in range(8):\n                if crc & 0x8000:\n                    crc = ((crc << 1) ^ poly) & 0xFFFF\n                else:\n                    crc = (crc << 1) & 0xFFFF\n        return crc\n\n    @staticmethod\n    def _calc_xor8(data: bytes) -> int:\n        """XOR-8 checksum: XOR of all bytes in the data range."""\n        result = 0\n        for b in data:\n            result ^= b\n        return result & 0xFF\n\n    # ------------------------------------------------------------------\n    #  Rule 1 – Constant Field\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def is_constant(cls, fragments: List[Any], width: int = 1) -> bool:\n        """\n        Paper Eq. (1): H(s) = 0 AND for all i, j: s_i ≡ s_j.\n        Zero Shannon entropy is equivalent to all values being identical.\n        Input: byte slices of any length.\n        """\n        if len(fragments) < 2:\n            return False\n\n        first = fragments[0]\n        if first is None:\n            return False\n\n        if isinstance(first, (bytes, bytearray)) and len(first) != width:\n            return False\n\n        for f in fragments[1:]:\n            if f is None or f != first:\n                return False\n        return True\n\n    # ------------------------------------------------------------------\n    #  Rule 2 – Sequence ID\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def is_sequence(cls, fragments: List[Any], width: int = 2) -> bool:\n        """\n        Paper Eq. (2):\n            for all i in [1, n-1]:  delta = v_{i+1} - v_i = delta\n            AND  v_i in [0, 2^(8k) - 1]   (no wrap-around)\n        k = byte length (1-4).  delta is a positive constant step.\n        """\n        nums = cls._to_int_list(fragments, width)\n        if nums is None or len(nums) < 3:\n            return False\n\n        delta = nums[1] - nums[0]\n        if delta <= 0:                         # must be an increment\n            return False\n\n        for i in range(1, len(nums) - 1):\n            if nums[i + 1] - nums[i] != delta:\n                return False\n\n        return True\n\n    # ------------------------------------------------------------------\n    #  Rule 3 – Timestamp\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def is_timestamp(\n        cls,\n        fragments: List[Any],\n        width: int = 4,\n        capture_start: Optional[float] = None,\n        capture_end: Optional[float] = None,\n        timezone_offset: float = 0.0,\n    ) -> bool:\n        """\n        Paper Eq. (3):  t_candidate in T_cap +/- Delta\n            T_cap = [t_start, t_end]   (capture window)\n            Delta  = 86400 s           (one day tolerance)\n        Input: 4/8-byte slices plus capture time range.\n        Supports timezone offsets.\n\n        The capture window is REQUIRED per the paper. If it is not\n        provided the rule returns False (conservative – no semantic\n        evidence of a timestamp).\n        """\n        nums = cls._to_int_list(fragments, width)\n        if nums is None or len(nums) < 3:\n            return False\n\n        # Capture window is required by the paper.\n        if capture_start is None or capture_end is None:\n            return False\n\n        delta = 86400.0  # paper: +/- 86400 seconds\n\n        lo = capture_start - delta\n        hi = capture_end + delta\n\n        # Apply timezone offset: the raw integer is interpreted as\n        # (timestamp + timezone_offset) so we compare in that frame.\n        for v in nums:\n            adjusted = float(v) + timezone_offset\n            if not (lo <= adjusted <= hi):\n                return False\n\n        # Paper requires only that candidates fall inside T_cap ± Δ.\n        return True\n\n    # ------------------------------------------------------------------\n    #  Rule 4 – Sparse Value\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def is_sparse(cls, fragments: List[Any], width: int = 1) -> bool:\n        """\n        Paper Eq. (4):\n            |V_unique| / 2^(8k) <= 0.02   AND   0 not in V_unique\n            k in {1, 2}\n        Identifies underutilized non-zero fields (e.g., protocol flags).\n        """\n        if width not in (1, 2):\n            return False\n\n        nums = cls._to_int_list(fragments, width)\n        if nums is None or not nums:\n            return False\n\n        unique_vals = set(nums)\n\n        # 0 must not appear in the unique value set.\n        if 0 in unique_vals:\n            return False\n\n        unique_count = len(unique_vals)\n        max_possible = 2 ** (8 * width)\n        ratio = unique_count / float(max_possible)\n\n        return ratio <= 0.02\n\n    # ------------------------------------------------------------------\n    #  Rule 5 – Address (paired fields)\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def is_address(\n        cls,\n        fragments_f1: List[Any],\n        fragments_f2: List[Any],\n        width: int = 4,\n        direction_labels: Optional[List[Any]] = None,\n    ) -> bool:\n        """\n        Paper Eq. (5):\n            For all c in C:  s1^c ≡ s2^c   (direction equivalence)\n            AND  rho(s1, s2) <= -0.8      (Pearson correlation)\n        Input: adjacent 2-4 byte slices.\n\n        When direction_labels is provided (one label per message, e.g.\n        \'client\'/\'server\' or 0/1), the exact paper condition is evaluated:\n        within each direction class, field-1 values must all be identical,\n        field-2 values must all be identical, and the two classes must use\n        swapped address pairs.\n\n        When direction_labels is None, a reciprocal-pair heuristic is used\n        as a self-contained fallback: every (a, b) pair must have its\n        mirror (b, a) present, and the mapping must be bijective.\n        """\n        nums1 = cls._to_int_list(fragments_f1, width)\n        nums2 = cls._to_int_list(fragments_f2, width)\n\n        if nums1 is None or nums2 is None or len(nums1) < 3:\n            return False\n        if len(nums1) != len(nums2):\n            return False\n\n        # Non-zero variance to avoid division by zero in correlation.\n        std1 = np.std(nums1)\n        std2 = np.std(nums2)\n        if std1 == 0 or std2 == 0:\n            return False\n\n        # --- Condition 1: direction-based equivalence -----------------\n        if direction_labels is not None:\n            # Exact paper implementation using class labels.\n            if len(direction_labels) != len(nums1):\n                return False\n\n            # Group messages by direction class.\n            classes: Dict[Any, Tuple[List[int], List[int]]] = {}\n            for label, v1, v2 in zip(direction_labels, nums1, nums2):\n                if label not in classes:\n                    classes[label] = ([], [])\n                classes[label][0].append(v1)\n                classes[label][1].append(v2)\n\n            if len(classes) < 2:\n                return False\n\n            # Within each class: field-1 constant, field-2 constant.\n            class_pairs: Dict[Any, Tuple[int, int]] = {}\n            for label, (f1_vals, f2_vals) in classes.items():\n                if len(set(f1_vals)) != 1 or len(set(f2_vals)) != 1:\n                    return False\n                a = f1_vals[0]\n                b = f2_vals[0]\n                if a == b:\n                    return False\n                class_pairs[label] = (a, b)\n\n            # Direction equivalence: the two classes must have swapped\n            # address pairs.  i.e. class1 = (A, B) and class2 = (B, A).\n            pair_list = list(class_pairs.values())\n            for i in range(len(pair_list)):\n                for j in range(i + 1, len(pair_list)):\n                    a_i, b_i = pair_list[i]\n                    a_j, b_j = pair_list[j]\n                    if a_i != b_j or b_i != a_j:\n                        return False\n\n        else:\n            # Heuristic fallback: reciprocal-pair check.\n            pairs = list(zip(nums1, nums2))\n            pair_set = set(pairs)\n\n            if len(pair_set) < 2:\n                return False\n\n            map12: Dict[int, int] = {}\n            map21: Dict[int, int] = {}\n\n            for a, b in pair_set:\n                if a == b:\n                    return False\n                if a in map12 and map12[a] != b:\n                    return False\n                if b in map21 and map21[b] != a:\n                    return False\n                map12[a] = b\n                map21[b] = a\n\n            for a, b in pair_set:\n                if (b, a) not in pair_set:\n                    return False\n\n        # --- Condition 2: Pearson correlation <= -0.8 -----------------\n        corr_matrix = np.corrcoef(nums1, nums2)\n        rho = corr_matrix[0, 1]\n\n        return bool(rho <= -0.8)\n\n    # ------------------------------------------------------------------\n    #  Rule 6 – Checksum\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def is_checksum(\n        cls,\n        fragments: List[Any],\n        width: int = 2,\n        messages: Optional[List[bytes]] = None,\n        offset: int = 0,\n        strict: bool = True,\n    ) -> bool:\n        """\n        Paper Eq. (6):  exists A in A_set,  A(D) ≡ s\n        Paper\'s named algorithms A_set = {CRC-16, XOR-8}.\n        Input: 1/2/4-byte slices plus data range (full message + offset).\n\n        When strict=True (default) only the paper\'s named algorithms are\n        evaluated: XOR-8 for width 1, CRC-16 for width 2.\n        When strict=False, practical extensions are also tried: RFC 1071\n        Internet Checksum and CRC-32 for width 4.\n        """\n        if not fragments or messages is None or len(messages) != len(fragments):\n            return False\n\n        if width not in (1, 2, 4):\n            return False\n\n        raw_frags = []\n        for f in fragments:\n            if not isinstance(f, (bytes, bytearray)) or len(f) != width:\n                return False\n            raw_frags.append(bytes(f))\n\n        # ---- 1-byte: XOR-8 -----------------------------------------\n        if width == 1:\n            xor8_exclude = True\n            xor8_suffix = True\n            for msg, frag in zip(messages, raw_frags):\n                val = frag[0]\n                payload_ex = msg[:offset] + msg[offset + width:]\n                payload_suf = msg[offset + width:]\n                if val != cls._calc_xor8(payload_ex):\n                    xor8_exclude = False\n                if val != cls._calc_xor8(payload_suf):\n                    xor8_suffix = False\n            return xor8_exclude or xor8_suffix\n\n        # ---- 2-byte: CRC-16 (paper) + RFC 1071 (extension) ------------\n        if width == 2:\n            crc16_match = True\n            rfc1071_match = True\n            for msg, frag in zip(messages, raw_frags):\n                val_big = int.from_bytes(frag, \'big\')\n\n                # CRC-16/CCITT over payload excluding the checksum field.\n                payload_ex = msg[:offset] + msg[offset + width:]\n                calc_crc16 = cls._calc_crc16_ccitt(payload_ex)\n                if val_big != calc_crc16:\n                    crc16_match = False\n\n                if not strict:\n                    # RFC 1071: zero out the checksum field then compute.\n                    zeroed_msg = msg[:offset] + b\'\\x00\\x00\' + msg[offset + width:]\n                    calc_rfc = cls._calc_internet_checksum(zeroed_msg)\n                    if val_big != calc_rfc:\n                        rfc1071_match = False\n\n            if strict:\n                return crc16_match\n            return crc16_match or rfc1071_match\n\n        # ---- 4-byte: CRC-32 (extension only) ------------------------\n        if width == 4:\n            if strict:\n                return False   # paper has no 4-byte algorithm\n\n            crc32_match_exclude = True\n            crc32_match_suffix = True\n            for msg, frag in zip(messages, raw_frags):\n                val_big = int.from_bytes(frag, \'big\')\n                val_little = int.from_bytes(frag, \'little\')\n\n                payload_ex = msg[:offset] + msg[offset + width:]\n                payload_suf = msg[offset + width:]\n\n                crc_ex = zlib.crc32(payload_ex) & 0xFFFFFFFF\n                crc_suf = zlib.crc32(payload_suf) & 0xFFFFFFFF\n\n                if val_big != crc_ex and val_little != crc_ex:\n                    crc32_match_exclude = False\n                if val_big != crc_suf and val_little != crc_suf:\n                    crc32_match_suffix = False\n\n            return crc32_match_exclude or crc32_match_suffix\n\n        return False\n\n    # ------------------------------------------------------------------\n    #  Rule registry\n    # ------------------------------------------------------------------\n\n    RULE_REGISTRY: List[Dict[str, Any]] = [\n\n        {\n            "name": "constant",\n            "widths": [8, 4, 2, 1],\n            "type": "single",\n            "func": lambda frags, w, X, offset:\n                SemanticRules.is_constant(frags, width=w)\n        },\n\n        {\n            "name": "sequence",\n            "widths": [1, 2, 3, 4],\n            "type": "single",\n            "func": lambda frags, w, X, offset:\n                SemanticRules.is_sequence(frags, width=w)\n        },\n\n        {\n            "name": "timestamp",\n            "widths": [4, 8],\n            "type": "single",\n            # NOTE: identify_boundary special-cases this rule by name to\n            # pass capture_start/capture_end/timezone_offset. The lambda\n            # below is used only for standalone calls without metadata.\n            "func": lambda frags, w, X, offset:\n                SemanticRules.is_timestamp(frags, width=w)\n        },\n\n        {\n            "name": "sparse",\n            "widths": [1, 2],\n            "type": "single",\n            "func": lambda frags, w, X, offset:\n                SemanticRules.is_sparse(frags, width=w)\n        },\n\n        {\n            "name": "address",\n            "widths": [2, 3, 4],\n            "type": "pair",\n            "func": lambda f1, f2, w, X, offset, direction_labels=None:\n                SemanticRules.is_address(f1, f2, width=w, direction_labels=direction_labels)\n        },\n\n        {\n            "name": "checksum",\n            "widths": [1, 2],\n            "type": "single",\n            "func": lambda frags, w, X, offset:\n                SemanticRules.is_checksum(\n                    fragments=frags, width=w, messages=X, offset=offset,\n                    strict=True\n                )\n        },\n    ]\n\n    # ------------------------------------------------------------------\n    #  Shared Semantic Region Scanner\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def _scan_semantic_regions(\n        cls,\n        X: List[bytes],\n        capture_start: Optional[float] = None,\n        capture_end: Optional[float] = None,\n        timezone_offset: float = 0.0,\n        direction_labels: Optional[List[Any]] = None,\n    ) -> List[Dict[str, Any]]:\n        """\n        Shared scanner used by both identify_boundary and\n        collect_semantic_regions to avoid logic drift.\n\n        Scans ALL offsets 0..min_len-1. At each offset, tries every rule;\n        on the first rule that matches, records a semantic region dict\n        and stops trying further rules at that offset (first-match precedence).\n\n        Returns a list of regions:\n            {"name": str, "offset": int, "width": int}\n        where width is the FULL span of the matched field (2*width for\n        pair rules like address).\n        """\n        if not X:\n            return []\n\n        min_len = min(len(m) for m in X)\n        regions: List[Dict[str, Any]] = []\n\n        for offset in range(min_len):\n\n            matched_at_offset = False\n\n            for rule in cls.RULE_REGISTRY:\n\n                if matched_at_offset:\n                    break\n\n                for width in rule["widths"]:\n\n                    # ---- Single-field rules ----\n                    if rule["type"] == "single":\n\n                        if offset + width > min_len:\n                            continue\n\n                        fragments = cls._extract_fragments(X, offset, width)\n                        if fragments is None:\n                            continue\n\n                        if rule["name"] == "timestamp":\n                            matched = cls.is_timestamp(\n                                fragments,\n                                width=width,\n                                capture_start=capture_start,\n                                capture_end=capture_end,\n                                timezone_offset=timezone_offset,\n                            )\n                        else:\n                            matched = rule["func"](fragments, width, X, offset)\n\n                        if matched:\n                            regions.append({\n                                "name": rule["name"],\n                                "offset": offset,\n                                "width": width,\n                            })\n                            matched_at_offset = True\n                            break\n\n                    # ---- Pair-field rule (Address) ----\n                    elif rule["type"] == "pair":\n\n                        if offset + 2 * width > min_len:\n                            continue\n\n                        fragments_1 = cls._extract_fragments(X, offset, width)\n                        fragments_2 = cls._extract_fragments(\n                            X, offset + width, width\n                        )\n\n                        if fragments_1 is None or fragments_2 is None:\n                            continue\n\n                        matched = rule["func"](\n                            fragments_1, fragments_2, width, X, offset,\n                            direction_labels=direction_labels,\n                        )\n\n                        if matched:\n                            regions.append({\n                                "name": rule["name"],\n                                "offset": offset,\n                                "width": 2 * width,\n                            })\n                            matched_at_offset = True\n                            break\n\n        return regions\n\n    # ------------------------------------------------------------------\n    #  Algorithm 1 – FOR-NFOR Boundary Detection\n    # ------------------------------------------------------------------\n\n    @classmethod\n    def identify_boundary(\n        cls,\n        X: List[bytes],\n        capture_start: Optional[float] = None,\n        capture_end: Optional[float] = None,\n        timezone_offset: float = 0.0,\n        direction_labels: Optional[List[Any]] = None,\n    ) -> Tuple[int, Set[int]]:\n        """\n        RPKClust Algorithm 1: FOR-NFOR Boundary Detection.\n\n        Scans ALL offsets 0..min_len-1. At each offset, tries every rule;\n        on the first rule that matches, records the right-boundary endpoint\n        {offset + l_r - 1} in hit_offsets and stops trying further rules at\n        that offset (first-match precedence). The outer loop continues to\n        the next offset. Final boundary B = max(hit_offsets) + 1.\n\n        Input:  Message set M, semantic rule library R (from RULE_REGISTRY),\n                optional capture window [capture_start, capture_end] and\n                timezone_offset for timestamp rule.\n        Output: FOR-NFOR boundary B\n        """\n        regions = cls._scan_semantic_regions(\n            X,\n            capture_start=capture_start,\n            capture_end=capture_end,\n            timezone_offset=timezone_offset,\n            direction_labels=direction_labels,\n        )\n\n        if not regions:\n            return 0, set()\n\n        hit_offsets: Set[int] = set()\n        for r in regions:\n            hit_offsets.add(r["offset"] + r["width"] - 1)\n\n        boundary_B = max(hit_offsets) + 1\n        return boundary_B, hit_offsets\n\n    @classmethod\n    def collect_semantic_regions(\n        cls,\n        X: List[bytes],\n        capture_start: Optional[float] = None,\n        capture_end: Optional[float] = None,\n        timezone_offset: float = 0.0,\n        direction_labels: Optional[List[Any]] = None,\n    ) -> List[Dict[str, Any]]:\n        """\n        Collect typed semantic regions from boundary scanning.\n\n        Returns a list of dicts: {"name", "offset", "width"}.\n        - Single-field rules have width = rule width (1, 2, 4, 8).\n        - Pair-field rules (address) have width = 2 * rule width.\n\n        These regions are needed by Algorithm 2 (extract_for_candidates)\n        to construct E_sem, F_sparse, and the scanning sequence S.\n        """\n        return cls._scan_semantic_regions(\n            X,\n            capture_start=capture_start,\n            capture_end=capture_end,\n            timezone_offset=timezone_offset,\n            direction_labels=direction_labels,\n        )', 'rpkclust/utils.py': 'from typing import List, Dict, Any, Set, Tuple, Optional, Callable\nimport numpy as np\n\nSemanticRegion = Dict[str, Any]\nFORCandidate = Dict[str, Any]\nSPARSE_RULES = ("sparse",)\n\n\ndef _region_offsets(region: SemanticRegion) -> Set[int]:\n    """Return the set of byte offsets covered by a semantic region [offset, offset+width-1]."""\n    start = region["offset"]\n    end = start + region["width"]\n    return set(range(start, end))\n\n\ndef _is_continuous(\n    s: int,\n    L: int,\n    excluded_offsets: Set[int],\n    max_offset: int,\n) -> bool:\n    """\n    Paper Line 7: IsContinuous(FFOR, s, L)\n    Checks that the interval [s, s+L-1] is fully inside the FOR\n    and does NOT overlap any excluded semantic offset.\n    """\n    if s + L > max_offset:\n        return False\n    for pos in range(s, s + L):\n        if pos in excluded_offsets:\n            return False\n    return True\n\n\ndef extract_for_candidates(\n    X: List[bytes],\n    boundary_B: int,\n    semantic_regions: Optional[List[SemanticRegion]] = None,\n    candidate_lengths: Tuple[int, ...] = (1, 2, 4),\n) -> List[FORCandidate]:\n    """\n    Algorithm 2: Keyword Candidate Generation in FOR.\n\n    Parameters\n    ----------\n    X : List[bytes]\n        Message set M (raw bytes).\n    boundary_B : int\n        FOR-NFOR boundary from Algorithm 1.\n    semantic_regions : Optional[List[SemanticRegion]]\n        Typed semantic hits from boundary identification, e.g.:\n        {"name": "constant", "offset": 0, "width": 2}\n        {"name": "sparse",   "offset": 8, "width": 1}\n        Algorithm 2 requires E_sem — if None, a warning is emitted and\n        all FOR offsets become candidates (NOT paper-accurate).\n    candidate_lengths : Tuple[int, ...]\n        Variable sliding window widths L to try (paper: "variable sliding\n        window").  Default (1, 2, 4) covers common protocol field sizes.\n\n    Returns\n    -------\n    List[FORCandidate]\n        Each candidate dict: {"offset", "width", "values"}.\n        Only offsets that pass semantic filtering, modulo alignment, and\n        continuity checks are returned.\n    """\n    candidates: List[FORCandidate] = []\n\n    if X is None or len(X) == 0 or boundary_B <= 0:\n        return candidates\n    if not candidate_lengths or any(\n        not isinstance(length, int) or isinstance(length, bool) or length <= 0\n        for length in candidate_lengths\n    ):\n        raise ValueError("candidate_lengths must contain positive integers")\n\n    min_len = min(len(msg) for msg in X)\n    max_offset = min(boundary_B, min_len)\n\n    # ----------------------------------------------------------\n    # Lines 1–3: Set construction\n    # ----------------------------------------------------------\n\n    FOR = set(range(max_offset))                       # [0, |F_FOR| - 1]\n\n    if semantic_regions is None:\n        import warnings\n        warnings.warn(\n            "Algorithm 2 requires typed semantic_regions (E_sem). "\n            "Without it, all FOR offsets become candidates (NOT paper-accurate). "\n            "Pass regions like {\'name\': \'constant\', \'offset\': 0, \'width\': 2}.",\n            UserWarning,\n            stacklevel=2,\n        )\n        semantic_regions = []\n\n    # Partition regions into excluded (E_sem) and sparse (F_sparse).\n    # Paper Line 1: E ← E_sem \\ F_sparse\n    #   ALL non-sparse semantic regions are excluded by default.\n    #   Only sparse regions are re-introduced into the scanning sequence.\n    sparse_regions: List[SemanticRegion] = []\n    excluded_regions: List[SemanticRegion] = []\n\n    for region in semantic_regions:\n        name = region.get("name", "")\n        if name in SPARSE_RULES:\n            sparse_regions.append(region)\n        else:\n            excluded_regions.append(region)\n\n    # Line 1: E ← E_sem \\ F_sparse\n    #   E = byte offsets covered by excluded semantic regions\n    #   (sparse regions are NOT excluded — they are re-introduced in S).\n    #   Offset-based subtraction handles overlaps correctly: if a sparse\n    #   offset is also covered by a non-sparse region, it stays in E.\n    all_semantic_offsets: Set[int] = set()\n    sparse_offsets: Set[int] = set()\n    sparse_starts: Set[int] = set()\n\n    for region in excluded_regions:\n        all_semantic_offsets |= _region_offsets(region) & FOR\n\n    for region in sparse_regions:\n        offsets = _region_offsets(region) & FOR\n        sparse_offsets |= offsets\n        if offsets:\n            sparse_starts.add(region["offset"])\n\n    # Offset-based subtraction matches E_sem \\ F_sparse:\n    # sparse-covered offsets are removed from E, even if semantic\n    # detections overlap.\n    E = all_semantic_offsets - sparse_offsets\n\n    # F_sparse = starting offsets of sparse fields (bounded to FOR).\n    F_sparse: Set[int] = sparse_starts & FOR\n\n    # Line 2: U ← [0, |F_FOR| − 1] \\ E\n    #   Undetected offsets = FOR offsets not covered by excluded regions.\n    U: Set[int] = FOR - E\n\n    # Line 3: S ← U ∪ F_sparse\n    #   Scanning sequence = undetected offsets + sparse field starts,\n    #   bounded to the FOR.\n    S: Set[int] = (U | F_sparse) & FOR\n\n    # ----------------------------------------------------------\n    # Lines 4–11: Candidate generation\n    # ----------------------------------------------------------\n\n    for L in candidate_lengths:\n        for s in sorted(S):\n            # Universal bounds check: candidate must fit inside FOR.\n            if s + L > max_offset:\n                continue\n\n            # Line 6: s ≡ 0 (mod L)  — length-aligned offset\n            if s % L != 0:\n                continue\n\n            # Line 7: continuity check for L > 1\n            if L > 1 and not _is_continuous(s, L, E, max_offset):\n                continue\n\n            # Line 8: C ← C ∪ {s}\n            # Extract values for downstream use.\n            values = [msg[s:s + L] for msg in X]\n\n            candidates.append({\n                "type": "FOR",\n                "tag": f"FOR_Offset_{s}_W{L}",\n                "offset": s,\n                "width": L,\n                "values": values,\n            })\n\n    # Deduplicate (same (offset, width) could arise from multiple L values\n    # or sparse re-introduction).\n    seen: Set[Tuple[int, int]] = set()\n    deduped: List[FORCandidate] = []\n    for c in candidates:\n        key = (c["offset"], c["width"])\n        if key not in seen:\n            seen.add(key)\n            deduped.append(c)\n\n    return deduped\n\nTLVRecord = Dict[str, Any]\nValidator = Callable[[bytes, int, int, int, int, int, bytes], bool]\n\n\ndef _parse_tlv_at(\n    m: bytes,\n    offset: int,\n    t_len: int,\n    l_len: int,\n) -> Optional[TLVRecord]:\n    """\n    Parse one TLV at the given offset within an NFOR message slice.\n    Returns None if the TLV header or value overflows the message\n    (paper Lines 8–11: data integrity check).\n    """\n    header_end = offset + t_len + l_len\n    if header_end > len(m):\n        return None\n\n    type_bytes = m[offset: offset + t_len]\n    len_bytes = m[offset + t_len: header_end]\n\n    type_val = int.from_bytes(type_bytes, "big")\n    len_val = int.from_bytes(len_bytes, "big")\n\n    value_start = header_end\n    value_end = value_start + len_val\n\n    # Paper Line 8: overflow check.\n    if value_end > len(m):\n        return None\n\n    value_bytes = m[value_start:value_end]\n\n    # Paper: T-V combined as keyword candidate.\n    tv_bytes = type_bytes + value_bytes\n\n    # Paper Line 15: P stores m[start:end] (full TLV segment).\n    tlv_bytes = m[offset:value_end]\n\n    return {\n        "start": offset,\n        "end": value_end,\n        "type_val": type_val,\n        "len_val": len_val,\n        "type_bytes": type_bytes,\n        "len_bytes": len_bytes,\n        "value_bytes": value_bytes,\n        "tv_bytes": tv_bytes,\n        "tlv_bytes": tlv_bytes,\n    }\n\n\ndef _default_validate_tlv(\n    m: bytes,\n    offset: int,\n    t_len: int,\n    l_len: int,\n    type_val: int,\n    len_val: int,\n    value_bytes: bytes,\n) -> bool:\n    """\n    Generic fallback for ValidateTLV (paper Line 12).\n    The paper does not define ValidateTLV in detail — it is a semantic\n    checker that confirms valid encoding beyond the structural overflow\n    check. Protocol-specific validation should be injected when available.\n\n    Default: accept any structurally valid TLV (already passed overflow check).\n    """\n    return True\n\n\ndef _detect_repeated_tlv(\n    m: bytes,\n    offset: int,\n    t_len: int,\n    l_len: int,\n    validate_tlv: Validator,\n) -> Optional[TLVRecord]:\n    """\n    Paper Line 16: DetectRepeatedTLV(m, end, t_len, l_len).\n    A repeated TLV exists if another valid TLV begins exactly at `offset`\n    (immediately after the previous one). This extends boundaries for\n    consecutive repeated TLV patterns.\n    """\n    rec = _parse_tlv_at(m, offset, t_len, l_len)\n    if rec is None:\n        return None\n\n    ok = validate_tlv(\n        m,\n        offset,\n        t_len,\n        l_len,\n        rec["type_val"],\n        rec["len_val"],\n        rec["value_bytes"],\n    )\n    if not ok:\n        return None\n\n    return rec\n\n\ndef extract_nfor_tlv_patterns(\n    X: List[bytes],\n    boundary_B: int,\n    t_len: int = 1,\n    l_len: int = 1,\n    validate_tlv: Optional[Validator] = None,\n    include_repeated_in_P: bool = True,\n) -> Tuple[List[TLVRecord], List[Dict[str, Any]]]:\n    """\n    RPKClust Algorithm 3: Keyword candidate generation in NFOR.\n\n    Parameters\n    ----------\n    X : List[bytes]\n        Full message set M. NFOR portions are extracted internally\n        starting at boundary_B.\n    boundary_B : int\n        FOR-NFOR boundary from Algorithm 1. NFOR = msg[boundary_B:].\n    t_len : int\n        Type field length in bytes.\n    l_len : int\n        Length field length in bytes.\n    validate_tlv : Optional[Validator]\n        Semantic validation function (paper Line 12). If None, a default\n        that accepts any structurally valid TLV is used. Protocol-specific\n        validation should be injected when available.\n    include_repeated_in_P : bool\n        If True (default), repeated TLV members are also appended to P.\n        Strict Algorithm 3 only records the initial pattern in P (Line 15)\n        before the repetition loop. Set False for strict paper compliance.\n        The repeated members are always counted for B regardless.\n\n    Returns\n    -------\n    P : List[TLVRecord]\n        Detected TLV patterns. Each record contains:\n        - type_val, type_bytes, len_val, value_bytes, tv_bytes, tlv_bytes\n        - relative_start/end (within NFOR slice)\n        - absolute_start/end (within original message)\n        - message_index\n    B : List[Dict[str, Any]]\n        Repeated TLV sequence boundaries. Each entry contains:\n        - relative_start/end, absolute_start/end\n        - first_end (end of first TLV in the sequence)\n        - repeated_count\n        - types (list of type_vals in the sequence)\n        - bytes (raw bytes of the repeated sequence)\n    """\n    if validate_tlv is None:\n        validate_tlv = _default_validate_tlv\n\n    P: List[TLVRecord] = []\n    B: List[Dict[str, Any]] = []\n\n    if X is None or len(X) == 0:\n        return P, B\n\n    if (\n        not isinstance(t_len, int) or isinstance(t_len, bool) or t_len <= 0\n        or not isinstance(l_len, int) or isinstance(l_len, bool) or l_len <= 0\n    ):\n        raise ValueError("t_len and l_len must be positive integers")\n\n    if boundary_B < 0:\n        boundary_B = 0\n\n    # ----------------------------------------------------------\n    # Paper Line 2: max_len ← min{len(m) | m ∈ M_NFOR}\n    # Paper input is M_NFOR, so slice full messages into NFOR portions.\n    # ----------------------------------------------------------\n\n    M_NFOR: List[Tuple[int, bytes]] = []\n    for msg_idx, msg in enumerate(X):\n        if boundary_B >= len(msg):\n            M_NFOR.append((msg_idx, b""))\n        else:\n            M_NFOR.append((msg_idx, msg[boundary_B:]))\n\n    max_len = min((len(m) for _, m in M_NFOR), default=0)\n\n    # ----------------------------------------------------------\n    # Paper Lines 3–27: Main loop\n    # ----------------------------------------------------------\n\n    for msg_idx, m in M_NFOR:\n\n        # Paper Line 4: offset ← 0 (relative to NFOR start)\n        offset = 0\n\n        # Paper Line 5: while offset ≤ len(m) − (t_len + l_len)\n        while offset <= len(m) - (t_len + l_len):\n\n            # Paper Lines 6–7: extract type and length fields\n            # Paper Lines 8–11: data integrity (overflow) check\n            rec = _parse_tlv_at(m, offset, t_len, l_len)\n\n            if rec is None:\n                # Paper Line 9: offset ← offset + 1\n                offset += 1\n                continue\n\n            # Paper Line 12: ValidateTLV semantic check\n            ok = validate_tlv(\n                m,\n                offset,\n                t_len,\n                l_len,\n                rec["type_val"],\n                rec["len_val"],\n                rec["value_bytes"],\n            )\n\n            if not ok:\n                # Paper Line 24: offset ← offset + 1\n                offset += 1\n                continue\n\n            # Paper Line 13: start ← offset\n            start = offset\n\n            # Paper Line 14: end ← offset + t_len + l_len + len_val\n            end = rec["end"]\n\n            # Paper Line 15: P ← P ∪ {(m[start:end], type_val)}\n            rec.update({\n                "message_index": msg_idx,\n                "relative_start": rec["start"],\n                "relative_end": rec["end"],\n                "absolute_start": boundary_B + rec["start"],\n                "absolute_end": boundary_B + rec["end"],\n            })\n            P.append(rec)\n\n            # Paper Lines 16–18: DetectRepeatedTLV loop\n            repeated_count = 0\n            repeated_types: List[int] = []\n\n            while True:\n                next_rec = _detect_repeated_tlv(\n                    m, end, t_len, l_len, validate_tlv,\n                )\n                if next_rec is None:\n                    break\n\n                repeated_count += 1\n                repeated_types.append(next_rec["type_val"])\n\n                # Downstream convenience: record repeated TLV in P.\n                # Strict Algorithm 3 only adds the initial pattern (Line 15).\n                if include_repeated_in_P:\n                    next_rec.update({\n                        "message_index": msg_idx,\n                        "relative_start": next_rec["start"],\n                        "relative_end": next_rec["end"],\n                        "absolute_start": boundary_B + next_rec["start"],\n                        "absolute_end": boundary_B + next_rec["end"],\n                        "is_repeated_member": True,\n                    })\n                    P.append(next_rec)\n\n                # Paper Line 17: end ← end + t_len + l_len + new_len_val\n                end = next_rec["end"]\n\n            # Paper Lines 19–21: store B for repetition counts >= 1.\n            # (Paper text: "For repetition counts >= 1, start-end positions\n            # are stored in B.")\n            if repeated_count > 0:\n                B.append({\n                    "message_index": msg_idx,\n                    "relative_start": start,\n                    "relative_end": end,\n                    "absolute_start": boundary_B + start,\n                    "absolute_end": boundary_B + end,\n                    "first_end": rec["end"],\n                    "repeated_count": repeated_count,\n                    "types": [rec["type_val"]] + repeated_types,\n                    "bytes": m[start:end],\n                })\n\n            # Paper Line 22: offset ← end\n            offset = end\n\n    return P, B\n\n\n# ------------------------------------------------------------------\n#  Aggregation wrapper (downstream convenience, NOT Algorithm 3)\n# ------------------------------------------------------------------\n\ndef extract_nfor_tlv_candidates(\n    X: List[bytes],\n    boundary_B: int,\n    t_len: int = 1,\n    l_len: int = 1,\n    validate_tlv: Optional[Validator] = None,\n) -> List[Dict[str, Any]]:\n    """\n    Aggregation wrapper that calls Algorithm 3 (extract_nfor_tlv_patterns)\n    and converts the output into a candidate list format compatible with\n    downstream keyword inference.\n\n    This is NOT part of Algorithm 3 — it is a convenience layer that\n    aggregates P by type_val across messages, using T-V combined values\n    (paper: "use T-V as the combined keyword field candidates").\n\n    Returns\n    -------\n    List[Dict[str, Any]]\n        Each candidate dict:\n        - tag_val: the TLV type value\n        - values: T-V combined bytes per message (None if absent)\n        - offsets: absolute offset per message (None if absent)\n        - patterns: all TLVRecords for this type\n        - repeated_boundaries: B entries involving this type\n    """\n    P, B = extract_nfor_tlv_patterns(\n        X, boundary_B, t_len=t_len, l_len=l_len, validate_tlv=validate_tlv,\n    )\n\n    # Group by type_val, keeping first occurrence per message.\n    by_type: Dict[int, Dict[int, TLVRecord]] = {}\n\n    for rec in P:\n        type_val = rec["type_val"]\n        msg_idx = rec["message_index"]\n\n        if type_val not in by_type:\n            by_type[type_val] = {}\n\n        if msg_idx not in by_type[type_val]:\n            by_type[type_val][msg_idx] = rec\n\n    candidates: List[Dict[str, Any]] = []\n    n = len(X)\n\n    for type_val, per_msg in sorted(by_type.items()):\n        values: List[Optional[bytes]] = []\n        offsets: List[Optional[int]] = []\n\n        for i in range(n):\n            if i in per_msg:\n                rec = per_msg[i]\n                # Paper: T-V combined as keyword candidate.\n                values.append(rec["tv_bytes"])\n                offsets.append(rec["absolute_start"])\n            else:\n                values.append(None)\n                offsets.append(None)\n\n        valid_count = sum(1 for v in values if v is not None)\n\n        candidates.append({\n            "type": "NFOR",\n            "tag": f"NFOR_TV_Type_{type_val}",\n            "tag_val": type_val,\n            "values": values,\n            "offsets": offsets,\n            "valid_count": valid_count,\n            "patterns": list(per_msg.values()),\n            "repeated_boundaries": [\n                b for b in B if type_val in b["types"]\n            ],\n        })\n\n    return candidates', 'datasets/__init__.py': 'from .generate_data import generate_generic_for_dataset, generate_generic_nfor_dataset\nfrom .stress_generator import BinaryProtocolStressGenerator, generate_stress_dataset\nfrom .dataset_loader import PcapDatasetLoader\n\n__all__ = [\n    "BinaryProtocolStressGenerator",\n    "PcapDatasetLoader",\n    "generate_generic_for_dataset",\n    "generate_generic_nfor_dataset",\n    "generate_stress_dataset",\n]', 'datasets/dataset_loader.py': '"""\nDataset loader for fetching and parsing real-world network packet captures (.pcap).\n"""\n\nimport os\nimport urllib.request\nimport struct\nfrom typing import Any, Dict, List, Tuple, Optional\nimport numpy as np\n\n\nclass PcapDatasetLoader:\n    """\n    Downloads and extracts raw application layer messages from classic PCAP files.\n    Supports Ethernet/IPv4 TCP and UDP frames using only the standard library.\n    """\n\n    # DEFAULT_URL = "https://raw.githubusercontent.com/wireshark/wireshark/master/test/captures/dhcp.pcap"\n    DEFAULT_URL = "https://mcfp.felk.cvut.cz/dataset/CTU-Malware-Capture-Botnet-42/botnet-capture-20110810-neris.pcap](https://mcfp.felk.cvut.cz/dataset/CTU-Malware-Capture-Botnet-42/botnet-capture-20110810-neris.pcap"\n\n    def __init__(self, target_dir: str = "datasets/downloads", allow_synthetic_fallback: bool = False):\n        self.target_dir = target_dir\n        self.allow_synthetic_fallback = allow_synthetic_fallback\n        # Per-payload metadata from the most recent extraction. It aligns\n        # positionally with the messages returned by extract_payloads().\n        self.last_metadata: List[Dict[str, Any]] = []\n        os.makedirs(self.target_dir, exist_ok=True)\n\n    def download_pcap(self, url: str = DEFAULT_URL, filename: str = "sample_protocol.pcap") -> str:\n        """Downloads a PCAP file if it does not already exist locally."""\n        file_path = os.path.join(self.target_dir, filename)\n        if not os.path.exists(file_path):\n            print(f"[PcapLoader] Fetching dataset from {url}...")\n            try:\n                urllib.request.urlretrieve(url, file_path)\n            except Exception as exc:\n                if os.path.exists(file_path):\n                    os.remove(file_path)\n                raise RuntimeError(f"Unable to download PCAP from {url}") from exc\n            print(f"[PcapLoader] Dataset saved to {file_path}")\n        else:\n            print(f"[PcapLoader] Using cached dataset at {file_path}")\n        return file_path\n\n    def extract_payloads(self, pcap_path: str, min_length: int = 8) -> Tuple[List[bytes], np.ndarray]:\n        """\n        Extracts UDP/TCP transport payloads (raw application binary messages) \n        from a global-header PCAP file.\n        """\n        payloads: List[bytes] = []\n        labels: List[int] = []\n        metadata: List[Dict[str, Any]] = []\n        session_initiators: Dict[str, Tuple[str, int]] = {}\n\n        with open(pcap_path, "rb") as f:\n            header = f.read(24)\n            if len(header) < 24:\n                raise ValueError("Invalid PCAP file: Header too short.")\n\n            magic_number = header[:4]\n            # Classic PCAP supports microsecond and nanosecond timestamps.\n            if magic_number == b"\\xa1\\xb2\\xc3\\xd4":\n                endian, timestamp_scale = ">", 1_000_000\n            elif magic_number == b"\\xa1\\xb2\\x3c\\x4d":\n                endian, timestamp_scale = ">", 1_000_000_000\n            elif magic_number == b"\\xd4\\xc3\\xb2\\xa1":\n                endian, timestamp_scale = "<", 1_000_000\n            elif magic_number == b"\\x4d\\x3c\\xb2\\xa1":\n                endian, timestamp_scale = "<", 1_000_000_000\n            else:\n                raise ValueError("Unsupported capture format; expected a classic PCAP file")\n            network = struct.unpack(f"{endian}I", header[20:24])[0]\n            if network != 1:  # DLT_EN10MB (Ethernet)\n                raise ValueError(f"Unsupported PCAP link type: {network}; only Ethernet is supported")\n\n            while True:\n                packet_hdr = f.read(16)\n                if len(packet_hdr) < 16:\n                    break\n                \n                ts_sec, ts_fraction, incl_len, _ = struct.unpack(f"{endian}IIII", packet_hdr)\n                pkt_data = f.read(incl_len)\n                if len(pkt_data) != incl_len:\n                    raise ValueError("Invalid PCAP file: truncated packet data")\n                transport = self._extract_transport_info(pkt_data)\n                if transport is not None and len(transport["payload"]) >= min_length:\n                    payload = transport["payload"]\n                    session_id = transport["session_id"]\n                    source = (transport["source_ip"], transport["source_port"])\n                    if session_id not in session_initiators:\n                        session_initiators[session_id] = self._infer_initiator(transport)\n                    direction = (\n                        "client"\n                        if source == session_initiators[session_id]\n                        else "server"\n                    )\n                    payloads.append(payload)\n                    metadata.append({\n                        "session_id": session_id,\n                        "direction": direction,\n                        "timestamp": ts_sec + ts_fraction / timestamp_scale,\n                        "source_ip": transport["source_ip"],\n                        "source_port": transport["source_port"],\n                        "destination_ip": transport["destination_ip"],\n                        "destination_port": transport["destination_port"],\n                        "protocol": transport["protocol"],\n                    })\n                    # Labels are only a convenience for benchmark captures, not protocol truth.\n                    labels.append(int(payload[0]))\n\n        if not payloads:\n            if self.allow_synthetic_fallback:\n                print("[PcapLoader] No qualifying payloads; generating explicit synthetic fallback traffic.")\n                payloads, labels_array = self._generate_fallback_traffic(min_length)\n                self.last_metadata = []\n                return payloads, labels_array\n            raise ValueError("No TCP or UDP application payloads meeting min_length were found")\n\n        self.last_metadata = metadata\n        return payloads, np.array(labels, dtype=int)\n\n    def extract_payloads_with_metadata(\n        self, pcap_path: str, min_length: int = 8\n    ) -> Tuple[List[bytes], np.ndarray, List[Dict[str, Any]]]:\n        """Extract payloads and their interaction metadata in one call."""\n        payloads, labels = self.extract_payloads(pcap_path, min_length)\n        return payloads, labels, self.last_metadata.copy()\n\n    @staticmethod\n    def _extract_transport_payload(packet: bytes) -> Optional[bytes]:\n        """Return a TCP or UDP payload from an Ethernet/IPv4 frame, if present."""\n        transport = PcapDatasetLoader._extract_transport_info(packet)\n        return None if transport is None else transport["payload"]\n\n    @staticmethod\n    def _extract_transport_info(packet: bytes) -> Optional[Dict[str, Any]]:\n        """Parse an Ethernet/IPv4 TCP or UDP frame and retain flow details."""\n        if len(packet) < 14:\n            return None\n        ethertype = int.from_bytes(packet[12:14], "big")\n        offset = 14\n        while ethertype in (0x8100, 0x88A8):\n            if len(packet) < offset + 4:\n                return None\n            ethertype = int.from_bytes(packet[offset + 2:offset + 4], "big")\n            offset += 4\n        if ethertype != 0x0800 or len(packet) < offset + 20:\n            return None\n        version_ihl = packet[offset]\n        if version_ihl >> 4 != 4:\n            return None\n        ip_header_len = (version_ihl & 0x0F) * 4\n        if ip_header_len < 20 or len(packet) < offset + ip_header_len:\n            return None\n        total_length = int.from_bytes(packet[offset + 2:offset + 4], "big")\n        ip_end = min(len(packet), offset + total_length) if total_length else len(packet)\n        protocol_number = packet[offset + 9]\n        protocol = {6: "tcp", 17: "udp"}.get(protocol_number)\n        if protocol is None:\n            return None\n        transport_offset = offset + ip_header_len\n        min_header_len = 20 if protocol == "tcp" else 8\n        if ip_end < transport_offset + min_header_len:\n            return None\n        source_ip = ".".join(str(value) for value in packet[offset + 12:offset + 16])\n        destination_ip = ".".join(str(value) for value in packet[offset + 16:offset + 20])\n        source_port = int.from_bytes(packet[transport_offset:transport_offset + 2], "big")\n        destination_port = int.from_bytes(packet[transport_offset + 2:transport_offset + 4], "big")\n        tcp_syn = False\n        tcp_ack = False\n        if protocol == "tcp":\n            header_len = (packet[transport_offset + 12] >> 4) * 4\n            if header_len < 20 or ip_end < transport_offset + header_len:\n                return None\n            payload_offset = transport_offset + header_len\n            flags = packet[transport_offset + 13]\n            tcp_syn, tcp_ack = bool(flags & 0x02), bool(flags & 0x10)\n        else:\n            payload_offset = transport_offset + 8\n        endpoints = sorted(((source_ip, source_port), (destination_ip, destination_port)))\n        session_id = f"{protocol}:{endpoints[0][0]}:{endpoints[0][1]}-{endpoints[1][0]}:{endpoints[1][1]}"\n        return {\n            "payload": packet[payload_offset:ip_end],\n            "source_ip": source_ip,\n            "source_port": source_port,\n            "destination_ip": destination_ip,\n            "destination_port": destination_port,\n            "protocol": protocol,\n            "session_id": session_id,\n            "tcp_syn": tcp_syn,\n            "tcp_ack": tcp_ack,\n        }\n\n    @staticmethod\n    def _infer_initiator(transport: Dict[str, Any]) -> Tuple[str, int]:\n        """Infer the client endpoint without fabricating missing capture facts."""\n        source = (transport["source_ip"], transport["source_port"])\n        destination = (transport["destination_ip"], transport["destination_port"])\n        if transport["protocol"] == "tcp" and transport["tcp_syn"] and not transport["tcp_ack"]:\n            return source\n        # For UDP, and TCP captures starting after the handshake, a privileged\n        # port is usually the service endpoint. Otherwise use first observed\n        # direction and retain the resulting inference in the documentation.\n        if source[1] <= 1024 < destination[1]:\n            return destination\n        return source\n\n    @staticmethod\n    def _generate_fallback_traffic(min_length: int) -> Tuple[List[bytes], np.ndarray]:\n        """Generate deterministic demo traffic when explicitly requested."""\n        rng = np.random.default_rng(42)\n        payloads, labels = [], []\n        for _ in range(300):\n            msg_type = int(rng.choice([0x01, 0x02, 0x05]))\n            body = bytes([msg_type, 0x00, 0x04]) + rng.bytes(max(16, min_length - 3))\n            payloads.append(body)\n            labels.append(msg_type)\n        return payloads, np.array(labels, dtype=int)', 'datasets/extract_data.py': '"""\nAMQP PCAP extractor for RPKClust evaluation.\n\nInput:\n    .pcap file\n\nOutput:\n    X:\n        List[bytes]\n        Raw AMQP frames suitable for RPKClust\n\nMetadata:\n        Connection information\n        Frame type\n        Channel\n        Payload size\n        Direction\n"""\n\nfrom dataclasses import dataclass\nfrom collections import defaultdict\nfrom typing import List, Dict, Tuple\n\nfrom scapy.all import (\n    rdpcap,\n    TCP,\n    Raw,\n    IP\n)\n\n\n@dataclass\nclass AMQPFrameMetadata:\n    connection: Tuple\n    frame_type: int\n    channel: int\n    payload_length: int\n    raw_length: int\n\n\nclass AMQPExtractor:\n    """\n    Extracts AMQP 0-9-1 frames from PCAP files.\n\n    Designed for protocol clustering algorithms such as RPKClust.\n\n    The output messages are raw binary protocol messages.\n    """\n\n    AMQP_PORT = 5672\n    FRAME_END = 0xCE\n\n    AMQP_HEADER = b"AMQP"\n\n    FRAME_TYPES = {\n        1: "METHOD",\n        2: "HEADER",\n        3: "BODY",\n        8: "HEARTBEAT"\n    }\n\n\n    def __init__(\n        self,\n        pcap_file: str,\n        client_only=True,\n        remove_handshake=True,\n        remove_heartbeat=True\n    ):\n\n        self.pcap_file = pcap_file\n\n        self.client_only = client_only\n        self.remove_handshake = remove_handshake\n        self.remove_heartbeat = remove_heartbeat\n\n\n        self.streams = {}\n\n        self.frames = []\n\n        self.metadata = []\n        self._extracted = False\n\n\n    # -----------------------------------------------------\n    # TCP extraction\n    # -----------------------------------------------------\n\n    def _extract_tcp_streams(self):\n\n        packets = rdpcap(self.pcap_file)\n\n\n        streams = defaultdict(bytearray)\n\n\n        for pkt in packets:\n\n\n            if not (\n                IP in pkt\n                and TCP in pkt\n                and Raw in pkt\n            ):\n                continue\n\n\n            tcp = pkt[TCP]\n\n\n            # client -> server\n            if self.client_only:\n\n                if tcp.dport != self.AMQP_PORT:\n                    continue\n\n\n            else:\n\n                if (\n                    tcp.sport != self.AMQP_PORT\n                    and\n                    tcp.dport != self.AMQP_PORT\n                ):\n                    continue\n\n\n\n            key = (\n                pkt[IP].src,\n                tcp.sport,\n                pkt[IP].dst,\n                tcp.dport\n            )\n\n\n            streams[key].extend(\n                bytes(pkt[Raw].load)\n            )\n\n\n        self.streams = dict(streams)\n\n        return self.streams\n\n\n\n    # -----------------------------------------------------\n    # AMQP frame parsing\n    # -----------------------------------------------------\n\n    def _parse_frames(\n        self,\n        stream: bytes,\n        connection\n    ):\n\n        frames = []\n\n\n        offset = 0\n\n\n        while offset + 7 < len(stream):\n\n\n            # Skip the eight-byte AMQP protocol header only when requested.\n            if self.remove_handshake and stream[offset:offset+4] == self.AMQP_HEADER:\n                offset += 8\n                continue\n\n\n\n            frame_type = stream[offset]\n\n\n            channel = int.from_bytes(\n                stream[\n                    offset+1:\n                    offset+3\n                ],\n                "big"\n            )\n\n\n            payload_size = int.from_bytes(\n                stream[\n                    offset+3:\n                    offset+7\n                ],\n                "big"\n            )\n\n\n            frame_end = (\n                offset\n                +\n                7\n                +\n                payload_size\n                +\n                1\n            )\n\n\n            if frame_end > len(stream):\n                break\n\n\n\n            frame = stream[offset:frame_end]\n\n\n            if frame[-1] != self.FRAME_END:\n\n                offset += 1\n                continue\n\n\n\n            # remove heartbeat frames\n            if (\n                self.remove_heartbeat\n                and frame_type == 8\n            ):\n\n                offset = frame_end\n                continue\n\n\n\n            frames.append(frame)\n\n\n            self.metadata.append(\n                AMQPFrameMetadata(\n                    connection=connection,\n                    frame_type=frame_type,\n                    channel=channel,\n                    payload_length=payload_size,\n                    raw_length=len(frame)\n                )\n            )\n\n\n            offset = frame_end\n\n\n\n        return frames\n\n\n\n    # -----------------------------------------------------\n    # Public extraction API\n    # -----------------------------------------------------\n\n    def extract(self):\n\n        # Extraction is repeatable: metadata must describe only this run.\n        self.frames = []\n        self.metadata = []\n        self._extracted = False\n        self._extract_tcp_streams()\n\n\n        all_frames = []\n\n\n        for connection, stream in self.streams.items():\n\n\n            frames = self._parse_frames(\n                bytes(stream),\n                connection\n            )\n\n\n            all_frames.extend(frames)\n\n\n\n        self.frames = all_frames\n        self._extracted = True\n\n\n        return self.frames\n\n\n\n    def get_messages(self):\n\n        """\n        Returns output compatible with:\n\n            RPKClust.fit(X)\n\n        """\n\n        if not self._extracted:\n\n            self.extract()\n\n\n        return self.frames\n\n\n\n    # -----------------------------------------------------\n    # Diagnostics\n    # -----------------------------------------------------\n\n    def summary(self):\n\n        if not self._extracted:\n\n            self.extract()\n\n\n        print("="*60)\n        print("AMQP EXTRACTION SUMMARY")\n        print("="*60)\n\n\n        print(\n            "TCP Streams:",\n            len(self.streams)\n        )\n\n\n        print(\n            "AMQP Frames:",\n            len(self.frames)\n        )\n\n\n        print("\\nFrame Types:")\n\n\n        counts = defaultdict(int)\n\n\n        for m in self.metadata:\n\n            counts[\n                self.FRAME_TYPES.get(\n                    m.frame_type,\n                    str(m.frame_type)\n                )\n            ] += 1\n\n\n        for k,v in counts.items():\n\n            print(\n                f" {k:<12}: {v}"\n            )\n\n\n        print("\\nFrame Size Statistics:")\n\n        sizes = [\n            m.raw_length\n            for m in self.metadata\n        ]\n\n\n        if sizes:\n\n            print(\n                "Min:",\n                min(sizes)\n            )\n\n            print(\n                "Max:",\n                max(sizes)\n            )\n\n            print(\n                "Avg:",\n                sum(sizes)/len(sizes)\n            )\n\n\n\n    def get_metadata(self):\n\n        return self.metadata', 'datasets/generate_data.py': '"""\nRPKClust Generic Dataset Generator (generate_data.py)\nGenerates synthetic binary message traces for Fixed-Offset Region (FOR) \nand Non-Fixed-Offset Region (NFOR TLV) benchmark evaluation.\n"""\n\nimport struct\nfrom typing import List, Tuple, Dict, Any, Optional\nimport numpy as np\n\n\nclass GenericDatasetGenerator:\n    """\n    Object-oriented generator for synthetic protocol traffic datasets.\n    Provides structured ground-truth metadata alongside binary payloads.\n    """\n\n    def __init__(self, seed: int = 42):\n        self.seed = seed\n\n    def generate_for_dataset(\n        self, num_messages: int = 1000\n    ) -> Tuple[List[bytes], np.ndarray, Dict[str, Any]]:\n        """\n        Generates a generic Fixed-Offset Region (FOR) binary message dataset.\n\n        Structure (Total fixed header = 12 bytes):\n            - Magic Constant  (2B) : Offset 0..1 (0xABCD)\n            - Version         (1B) : Offset 2    (0x01)\n            - OpCode / Type   (1B) : Offset 3    [KEYWORD -> Cluster Ground Truth]\n            - Timestamp       (4B) : Offset 4..7 (Sequential integer)\n            - Sequence Num    (4B) : Offset 8..11 (Sequential integer)\n            - Variable Body   (4-16B): Random Payload\n\n        Returns:\n            X (List[bytes]): List of binary messages.\n            y (np.ndarray): Target cluster labels (derived from OpCode).\n            metadata (Dict[str, Any]): Ground-truth parameters for evaluation.\n        """\n        if not isinstance(num_messages, int) or isinstance(num_messages, bool) or num_messages < 0:\n            raise ValueError("num_messages must be a non-negative integer")\n        rng = np.random.default_rng(self.seed)\n        X: List[bytes] = []\n        y: List[int] = []\n        interaction_metadata: List[Dict[str, Any]] = []\n\n        opcodes = [0x01, 0x02, 0x03, 0x04]\n\n        for i in range(num_messages):\n            magic = 0xABCD\n            version = 1\n            # Adjacent messages model a request/response transaction and\n            # intentionally share an opcode for remote-coupling evaluation.\n            opcode = int(rng.choice(opcodes)) if i % 2 == 0 else y[-1]\n            timestamp = 10000 + i\n            sequence = i\n            \n            # Variable length tail payload\n            payload_len = int(rng.integers(4, 17))\n            payload = rng.bytes(payload_len)\n\n            # Pack fixed header (12 bytes total)\n            fixed_header = struct.pack(">HBBII", magic, version, opcode, timestamp, sequence)\n            msg = fixed_header + payload\n\n            X.append(msg)\n            y.append(opcode)\n            interaction_metadata.append({\n                "session_id": i // 2,\n                "direction": "client" if i % 2 == 0 else "server",\n                "timestamp": float(i),\n            })\n\n        metadata = {\n            "dataset_type": "FOR",\n            "true_boundary_B": 12,        # Fixed header portion\n            "true_keyword_offset": 3,     # OpCode byte index\n            "true_keyword_width": 1,\n            "num_clusters": len(opcodes),\n            "interaction_metadata": interaction_metadata,\n        }\n\n        return X, np.array(y, dtype=int), metadata\n\n    def generate_nfor_dataset(\n        self, num_messages: int = 1000\n    ) -> Tuple[List[bytes], np.ndarray, Dict[str, Any]]:\n        """\n        Generates a generic Non-Fixed-Offset Region (NFOR TLV) binary message dataset.\n\n        Header (Fixed Boundary B = 7 bytes):\n            - Magic Constant (2B) : Offset 0..1 (0xABCD)\n            - Version        (1B) : Offset 2    (0x01)\n            - Timestamp      (4B) : Offset 3..6\n\n        Body (Variable NFOR TLV Region):\n            - Command TLV  (Type=0x0A, Len=1, Value=[0x10, 0x20, 0x30, 0x40]) -> Target Keyword\n            - Data TLV     (Type=0x0B, Len=2..8, Data=random)\n            - Optional TLV (Type=0x0C, Len=1..4, Data=random) [30% occurrence]\n\n        Returns:\n            X (List[bytes]): List of binary messages.\n            y (np.ndarray): Target cluster labels (derived from Command TLV value).\n            metadata (Dict[str, Any]): Ground-truth parameters for evaluation.\n        """\n        if not isinstance(num_messages, int) or isinstance(num_messages, bool) or num_messages < 0:\n            raise ValueError("num_messages must be a non-negative integer")\n        rng = np.random.default_rng(self.seed)\n        X: List[bytes] = []\n        y: List[int] = []\n        interaction_metadata: List[Dict[str, Any]] = []\n\n        cmd_values = [0x10, 0x20, 0x30, 0x40]\n\n        for i in range(num_messages):\n            # Adjacent messages model a request/response transaction.\n            cmd_val = int(rng.choice(cmd_values)) if i % 2 == 0 else y[-1]\n\n            # Fixed Header (7 Bytes)\n            magic = 0xABCD\n            version = 1\n            timestamp = 20000 + i\n            header = struct.pack(">HBI", magic, version, timestamp)\n\n            # Required Command TLV (3 Bytes)\n            tlv_cmd = struct.pack(">BBB", 0x0A, 1, cmd_val)\n\n            # Required Data TLV (2 + data_len Bytes)\n            data_len = int(rng.integers(2, 9))\n            data = rng.bytes(data_len)\n            tlv_data = struct.pack(">BB", 0x0B, data_len) + data\n\n            tlvs = [tlv_cmd, tlv_data]\n\n            # Optional TLV (30% probability)\n            if rng.random() < 0.30:\n                opt_len = int(rng.integers(1, 5))\n                opt_data = rng.bytes(opt_len)\n                tlv_optional = struct.pack(">BB", 0x0C, opt_len) + opt_data\n                tlvs.append(tlv_optional)\n\n            # Shuffle TLV sequence in the NFOR body to induce position variance\n            rng.shuffle(tlvs)\n\n            msg = header + b"".join(tlvs)\n\n            X.append(msg)\n            y.append(cmd_val)\n            interaction_metadata.append({\n                "session_id": i // 2,\n                "direction": "client" if i % 2 == 0 else "server",\n                "timestamp": float(i),\n            })\n\n        metadata = {\n            "dataset_type": "NFOR",\n            "true_boundary_B": 7,\n            "keyword_tag": "TLV_Type_10",\n            "num_clusters": len(cmd_values),\n            "interaction_metadata": interaction_metadata,\n        }\n\n        return X, np.array(y, dtype=int), metadata\n\n\n# =====================================================================\n# Functional Wrapper Interface (Maintains Backward Compatibility)\n# =====================================================================\n\ndef generate_generic_for_dataset(\n    num_messages: int = 1000, seed: int = 54761161\n) -> Tuple[List[bytes], np.ndarray]:\n    """Functional wrapper for generating FOR datasets."""\n    generator = GenericDatasetGenerator(seed=seed)\n    X, y, _ = generator.generate_for_dataset(num_messages=num_messages)\n    return X, y\n\n\ndef generate_generic_nfor_dataset(\n    num_messages: int = 1000, seed: int = 841561854\n) -> Tuple[List[bytes], np.ndarray]:\n    """Functional wrapper for generating NFOR datasets."""\n    generator = GenericDatasetGenerator(seed=seed)\n    X, y, _ = generator.generate_nfor_dataset(num_messages=num_messages)\n    return X, y', 'datasets/stress_generator.py': '"""\nRPKClust Protocol Stress Generator (stress_generator.py)\nGenerates noisy, corrupted binary message traces to stress-test \nboundary identification and TLV candidate extraction under real-world noise.\n"""\n\nimport struct\nfrom typing import List, Tuple, Dict, Any\nimport numpy as np\n\n\nclass BinaryProtocolStressGenerator:\n    """\n    Generates noisy binary message traces (list[bytes]) to stress-test \n    RPKClust\'s boundary identification and TLV candidate extraction.\n    """\n\n    def __init__(\n        self,\n        num_messages: int = 1000,\n        noise_level: float = 0.2,\n        seed: int = 42,\n    ):\n        if not isinstance(num_messages, int) or isinstance(num_messages, bool) or num_messages < 0:\n            raise ValueError("num_messages must be a non-negative integer")\n        if not isinstance(noise_level, (int, float)) or not 0.0 <= noise_level <= 1.0:\n            raise ValueError("noise_level must be between 0.0 and 1.0")\n        self.num_messages = num_messages\n        self.noise_level = float(noise_level)\n        self.seed = seed\n        self.rng = np.random.default_rng(seed)\n\n    def generate(self) -> Tuple[List[bytes], np.ndarray]:\n        """\n        Generates noisy message list and target labels.\n        (Backward-compatible 2-tuple return).\n        """\n        X, y, _ = self.generate_with_metadata()\n        return X, y\n\n    def generate_with_metadata(\n        self,\n    ) -> Tuple[List[bytes], np.ndarray, Dict[str, Any]]:\n        """\n        Generates noisy message traces along with ground-truth metadata \n        for evaluation reporting.\n        """\n        X: List[bytes] = []\n        y: List[int] = []\n        interaction_metadata: List[Dict[str, Any]] = []\n        opcodes = [0x01, 0x02, 0x03, 0x04]\n\n        for i in range(self.num_messages):\n            opcode = int(self.rng.choice(opcodes)) if i % 2 == 0 else y[-1]\n            magic = 0xABCD\n            version = 1\n            sequence = i\n\n            # 1. Non-monotonic timestamp & sequence noise\n            if self.rng.random() < self.noise_level:\n                timestamp = int(self.rng.integers(0, 10000))\n            else:\n                timestamp = 1000 + i\n\n            if self.rng.random() < self.noise_level:\n                sequence = int(self.rng.integers(0, 5000))\n            else:\n                sequence = i\n\n            # 2. Fixed Header binary packing (12 Bytes total)\n            # Offset 0..1: Magic (2B)\n            # Offset 2   : Version (1B)\n            # Offset 3   : OpCode (1B) -> Target Keyword Offset = 3\n            # Offset 4..7: Timestamp (4B)\n            # Offset 8..11: Sequence (4B)\n            header = struct.pack(\n                ">HBBII",\n                magic,\n                version,\n                opcode,\n                timestamp,\n                sequence,\n            )\n\n            # 3. Dynamic / Corrupted TLV noise payload (NFOR portion)\n            if self.rng.random() < self.noise_level:\n                # Corrupted/Malformed TLV (random length/tag)\n                tlv_tag = int(self.rng.integers(0x80, 0xFF))\n                tlv_len = int(self.rng.integers(1, 10))\n                tlv_val = self.rng.bytes(tlv_len)\n                payload = struct.pack(">BB", tlv_tag, tlv_len) + tlv_val\n            else:\n                # Valid TLV Options\n                tlvs = [struct.pack(">BBB", 0x0A, 1, opcode)]\n\n                if self.rng.random() < 0.5:\n                    data = self.rng.bytes(4)\n                    tlvs.append(struct.pack(">BB", 0x0B, 4) + data)\n\n                self.rng.shuffle(tlvs)\n                payload = b"".join(tlvs)\n\n            msg = header + payload\n            X.append(msg)\n            y.append(opcode)\n            interaction_metadata.append({\n                "session_id": i // 2,\n                "direction": "client" if i % 2 == 0 else "server",\n                "timestamp": float(i),\n            })\n\n        metadata = {\n            "dataset_type": "Binary_Stress",\n            "true_boundary_B": 12,        # Fixed header length before variable TLVs\n            "true_keyword_offset": 3,     # Opcode byte location\n            "noise_level": self.noise_level,\n            "num_clusters": len(opcodes),\n            "interaction_metadata": interaction_metadata,\n        }\n\n        return X, np.array(y, dtype=int), metadata\n\n\n# =====================================================================\n# Functional Wrapper\n# =====================================================================\n\ndef generate_stress_dataset(\n    num_messages: int = 1000, noise_level: float = 0.2, seed: int = 42\n) -> Tuple[List[bytes], np.ndarray]:\n    """Functional helper to generate binary stress dataset directly."""\n    generator = BinaryProtocolStressGenerator(\n        num_messages=num_messages, noise_level=noise_level, seed=seed\n    )\n    return generator.generate()', 'experiments/__init__.py': '# Experiments package init', 'experiments/compare_baselines.py': 'import time\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans, DBSCAN, SpectralClustering\nfrom sklearn.mixture import GaussianMixture\n\nfrom rpkclust import RPKClust\nfrom rpkclust.metrics import convert_bytes_to_feature_matrix, evaluate_clustering\n\ndef run_baseline_comparison(messages, labels_true):\n    """\n    Runs RPKClust and standard baseline models on the exact same dataset,\n    returning a performance table.\n    """\n    if len(messages) != len(labels_true):\n        raise ValueError("messages and labels_true must have the same length")\n    if len(messages) < 2:\n        raise ValueError("at least two messages are required for baseline comparison")\n    X_mat = convert_bytes_to_feature_matrix(messages)\n    n_clusters = len(set(labels_true))\n    if n_clusters < 2 or n_clusters > len(messages):\n        raise ValueError("labels_true must contain between 2 and len(messages) clusters")\n    \n    results = []\n    \n    # 1. RPKClust\n    t0 = time.time()\n    rpk = RPKClust()\n    rpk_labels = rpk.fit_predict(messages)\n    t_rpk = time.time() - t0\n    rpk_res = evaluate_clustering(labels_true, rpk_labels, X_mat, t_rpk)\n    rpk_res["Model"] = "RPKClust (Ours)"\n    results.append(rpk_res)\n    \n    # 2. K-Means\n    t0 = time.time()\n    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)\n    km_labels = km.fit_predict(X_mat)\n    t_km = time.time() - t0\n    km_res = evaluate_clustering(labels_true, km_labels, X_mat, t_km)\n    km_res["Model"] = "K-Means"\n    results.append(km_res)\n    \n    # 3. DBSCAN\n    t0 = time.time()\n    db = DBSCAN(eps=150.0, min_samples=5)\n    db_labels = db.fit_predict(X_mat)\n    t_db = time.time() - t0\n    db_res = evaluate_clustering(labels_true, db_labels, X_mat, t_db)\n    db_res["Model"] = "DBSCAN"\n    results.append(db_res)\n    \n    # 4. Gaussian Mixture Model (GMM)\n    t0 = time.time()\n    gmm = GaussianMixture(n_components=n_clusters, random_state=42)\n    gmm_labels = gmm.fit_predict(X_mat)\n    t_gmm = time.time() - t0\n    gmm_res = evaluate_clustering(labels_true, gmm_labels, X_mat, t_gmm)\n    gmm_res["Model"] = "GMM"\n    results.append(gmm_res)\n    \n    # 5. Spectral Clustering\n    t0 = time.time()\n    spec = SpectralClustering(\n        n_clusters=n_clusters,\n        n_neighbors=min(10, len(messages) - 1),\n        random_state=42,\n        assign_labels="kmeans",\n    )\n    spec_labels = spec.fit_predict(X_mat)\n    t_spec = time.time() - t0\n    spec_res = evaluate_clustering(labels_true, spec_labels, X_mat, t_spec)\n    spec_res["Model"] = "Spectral"\n    results.append(spec_res)\n    \n    df_res = pd.DataFrame(results)\n    # Reorder columns for presentation\n    cols = ["Model", "ARI", "NMI", "V-Measure", "Silhouette", "Davies-Bouldin", "Execution Time (s)"]\n    return df_res[cols]', 'experiments/parameter_analysis.py': 'import os\nimport time\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom rpkclust import RPKClust\nfrom datasets.generate_data import generate_generic_for_dataset\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics import adjusted_rand_score\nfrom rpkclust.metrics import convert_bytes_to_feature_matrix\nfrom datasets.generate_data import generate_generic_nfor_dataset\n\ndef analyze_sample_size_scalability(sample_sizes=[100, 250, 500, 1000, 2000]):\n    """\n    Evaluates how RPKClust execution time scales with the number of messages N.\n    """\n    records = []\n    for n in sample_sizes:\n        messages, labels = generate_generic_for_dataset(num_messages=n)\n        \n        t0 = time.time()\n        model = RPKClust()\n        model.fit(messages)\n        t_elapsed = time.time() - t0\n        \n        records.append({\n            "Sample Size (N)": n,\n            "Execution Time (s)": t_elapsed,\n            "Boundary Found (B)": model.boundary_B,\n            "Best Candidate Prob": round(model.best_candidate[\'prob\'], 4) if model.best_candidate else 0.0\n        })\n        \n    return pd.DataFrame(records)\n\ndef analyze_offset_shift_impact(\n    max_pad_lengths=[0, 5, 10, 20, 50, 100], output_dir="results/figures"\n):\n    """\n    Demonstrates the exact problem RPKClust solves: Traditional algorithms fail \n    when the keyword offset shifts.\n    """\n    print("\\nRunning NFOR Offset Shift Impact Analysis...")\n    records = []\n    \n    for pad in max_pad_lengths:\n        # Insert padding after the 7-byte fixed header, so every command TLV\n        # is shifted by the recorded amount regardless of TLV ordering.\n        rng = np.random.default_rng(42)\n        m, l_true = generate_generic_nfor_dataset(num_messages=400)\n        m_shifted = []\n        for msg in m:\n            pad_len = int(rng.integers(0, pad + 1)) if pad > 0 else 0\n            m_shifted.append(msg[:7] + rng.bytes(pad_len) + msg[7:])\n            \n        # 1. RPKClust\n        rpk = RPKClust()\n        l_rpk = rpk.fit_predict(m_shifted)\n        ari_rpk = adjusted_rand_score(l_true, l_rpk)\n        \n        # 2. K-Means\n        X_mat = convert_bytes_to_feature_matrix(m_shifted)\n        km = KMeans(n_clusters=len(np.unique(l_true)), random_state=42, n_init=10)\n        l_km = km.fit_predict(X_mat)\n        ari_km = adjusted_rand_score(l_true, l_km)\n        \n        records.append({\n            "Max Shift (Bytes)": pad,\n            "RPKClust ARI": ari_rpk,\n            "K-Means ARI": ari_km\n        })\n        \n    df_impact = pd.DataFrame(records)\n    \n    # Plotting\n    plt.figure(figsize=(8, 5))\n    plt.plot(df_impact["Max Shift (Bytes)"], df_impact["RPKClust ARI"], marker=\'o\', label=\'RPKClust (Ours)\', color=\'blue\', linewidth=2)\n    plt.plot(df_impact["Max Shift (Bytes)"], df_impact["K-Means ARI"], marker=\'x\', label=\'K-Means\', color=\'red\', linestyle=\'--\', linewidth=2)\n    plt.title("Impact of NFOR Variable Offsets on Clustering Accuracy")\n    plt.xlabel("Maximum Variable Padding (Bytes)")\n    plt.ylabel("Adjusted Rand Index (ARI)")\n    plt.legend()\n    plt.grid(True, alpha=0.3)\n    plt.tight_layout()\n    os.makedirs(output_dir, exist_ok=True)\n    plt.savefig(os.path.join(output_dir, "offset_shift_impact.png"), dpi=300)\n    plt.close()\n    \n    return df_impact', 'experiments/run_rpkclust.py': 'import os\n\nimport matplotlib.pyplot as plt\nimport numpy as np\n\nfrom datasets import generate_generic_for_dataset, generate_generic_nfor_dataset\nfrom experiments.compare_baselines import run_baseline_comparison\nfrom rpkclust.metrics import convert_bytes_to_feature_matrix\n\ntry:  # Support both `python -m experiments.run_rpkclust` and direct execution.\n    from .parameter_analysis import (\n        analyze_offset_shift_impact,\n        analyze_sample_size_scalability,\n    )\nexcept ImportError:\n    from parameter_analysis import (\n        analyze_offset_shift_impact,\n        analyze_sample_size_scalability,\n    )\n\ndef run_all_experiments():\n    """Executes the complete experimental suite and exports visual artifacts."""\n    os.makedirs("results/figures", exist_ok=True)\n    os.makedirs("results/tables", exist_ok=True)\n    \n    print("==================================================")\n    print("       RUNNING RPKCLUST BENCHMARK SUITE          ")\n    print("==================================================")\n    \n    # 1. Benchmark on Dataset 1: Simple FOR\n    print("\\n[1/3] Benchmarking Dataset 1: Simple FOR (OpCode)...")\n    m1, l1 = generate_generic_for_dataset()\n    res_df1 = run_baseline_comparison(m1, l1)\n    res_df1.to_csv("results/tables/dataset1_for_results.csv", index=False)\n    print(res_df1.to_string(index=False))\n\n    generate_benchmark_barchart(res_df1, "Performance Comparison: Simple FOR", "benchmark_for_barchart.png")\n    \n    # 2. Benchmark on Dataset 2: NFOR TLV\n    print("\\n[2/3] Benchmarking Dataset 2: NFOR TLV (DHCP-style)...")\n    m2, l2 = generate_generic_nfor_dataset()\n    res_df2 = run_baseline_comparison(m2, l2)\n    res_df2.to_csv("results/tables/dataset2_nfor_results.csv", index=False)\n    print(res_df2.to_string(index=False))\n\n    generate_benchmark_barchart(res_df2, "Performance Comparison: NFOR TLV", "benchmark_nfor_barchart.png")\n    \n    \n    \n    # Scalability Plot\n    df_scale = analyze_sample_size_scalability()\n    df_scale.to_csv("results/tables/scalability_results.csv", index=False)\n    \n    plt.figure(figsize=(7, 4))\n    plt.plot(df_scale["Sample Size (N)"], df_scale["Execution Time (s)"], marker=\'o\', color=\'crimson\', linewidth=2)\n    plt.title("RPKClust Execution Time vs. Message Count (N)")\n    plt.xlabel("Number of Messages (N)")\n    plt.ylabel("Execution Time (seconds)")\n    plt.grid(True, linestyle="--", alpha=0.6)\n    plt.tight_layout()\n    plt.savefig("results/figures/rpkclust_scalability.png", dpi=300)\n    plt.close()\n    \n    # Run Variable Offset Stress Test for Question 1 & 5\n    print("\\n[+] Running Offset Shift Stress Test...")\n    df_impact = analyze_offset_shift_impact()\n    df_impact.to_csv("results/tables/offset_shift_impact.csv", index=False)\n    \n    print("\\nAll experiments finished successfully! Results saved to \'results/\'.")\n\ndef generate_benchmark_barchart(df_res, title, filename):\n    """Generates a grouped bar chart for slide presentations."""\n    models = df_res["Model"].tolist()\n    ari = df_res["ARI"].tolist()\n    nmi = df_res["NMI"].tolist()\n    \n    x = np.arange(len(models))\n    width = 0.35\n    \n    fig, ax = plt.subplots(figsize=(10, 6))\n    ax.bar(x - width/2, ari, width, label=\'ARI (Accuracy)\', color=\'#2ca02c\')\n    ax.bar(x + width/2, nmi, width, label=\'NMI (Information Info)\', color=\'#1f77b4\')\n    \n    ax.set_ylabel(\'Score (0.0 to 1.0)\')\n    ax.set_title(title)\n    ax.set_xticks(x)\n    ax.set_xticklabels(models, rotation=15)\n    ax.legend()\n    plt.tight_layout()\n    plt.savefig(f"results/figures/{filename}", dpi=300)\n    plt.close()\n\nif __name__ == "__main__":\n    run_all_experiments()'}


In [3]:
from pathlib import Path
import sys
import tempfile

WORKSPACE = Path(tempfile.mkdtemp(prefix='rpkclust_notebook_'))
for relative_path, source in SOURCES.items():
    destination = WORKSPACE / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')

sys.path.insert(0, str(WORKSPACE))
print('Reconstructed', len(SOURCES), 'source files in', WORKSPACE)


Reconstructed 18 source files in C:\Users\E5511~1.GHO\AppData\Local\Temp\rpkclust_notebook_0zzdaiio


## Quick synthetic run

This runs the full pipeline with capture metadata. It does not require downloading a PCAP.

In [4]:
from datasets.generate_data import GenericDatasetGenerator
from temp_evaluator import RPKClustEvaluator

generator = GenericDatasetGenerator(seed=54761161)
messages, labels, metadata = generator.generate_for_dataset(num_messages=100)
evaluator = RPKClustEvaluator(output_dir='results', fig_format='png', dpi=150)
summary = evaluator.run_diagnostics(
    messages,
    labels,
    dataset_name='Generic FOR',
    true_boundary=metadata['true_boundary_B'],
    true_keyword_offset=metadata['true_keyword_offset'],
    fit_kwargs={'interaction_metadata': metadata['interaction_metadata']},
)
summary



  RPKCLUST PAPER DIAGNOSTIC REPORT: Generic FOR
Identifying Boundaries...
  Boundary B = 16
  Semantic regions: 15
Extracting FOR Candidates...
  FOR candidates: 8
Extracting NFOR Candidates...
  NFOR candidates: 8
  Total candidates: 16
Two Stage Bayesian Inference...
Keyword Selection...
  Best: FOR_Offset_3_W1 prob=1.0000
Assigning Semantic Clustering...

[1] FOR-NFOR BOUNDARY IDENTIFICATION (Algorithm 1)
  Inferred FOR-NFOR Boundary (B) : 16 bytes
  Execution Time               : 0.2360 seconds
  Ground Truth Boundary        : 12 bytes
  Boundary Absolute Error      : 4 bytes
  Boundary Relative Error      : 33.33%

[2] REGION-PARTITIONED CANDIDATE GENERATION
  FOR Candidates Extracted     : 8
  NFOR Candidates Extracted    : 8
  Total Candidates Evaluated  : 16

[3] TWO-STAGE BAYESIAN INFERENCE RANKINGS (Top Candidates)
-----------------------------------------------------------------
Rank Tag                   Type  p_bit   p_offset  Stage1  Posterior P
-------------------------

{'Dataset': 'Generic FOR',
 'Messages': 100,
 'Boundary_Inferred': 16,
 'Boundary_True': 12,
 'Boundary_Error': 4,
 'FOR_Candidates': 8,
 'NFOR_Candidates': 8,
 'Total_Candidates': 16,
 'Keyword_Correct': True,
 'Keyword_Rank': 1,
 'Homogeneity': 1.0,
 'Completeness': 1.0,
 'V_Measure': 1.0,
 'ARI': 1.0,
 'NMI': 1.0,
 'Accuracy': 1.0,
 'Clusters_Found': 4,
 'Time_s': 0.236,
 'Memory_MB': 0.55}

## Full evaluation (optional)

The following cell executes the project entry point, including the external PCAP download. It can take substantially longer than the synthetic run.

In [5]:
# Uncomment to run every dataset and download the configured PCAP:
from main import main
main()


 Initializing RPKClust Paper-Faithful Evaluation Suite


  RPKCLUST PAPER DIAGNOSTIC REPORT: Generic FOR
Identifying Boundaries...
  Boundary B = 16
  Semantic regions: 14
Extracting FOR Candidates...
  FOR candidates: 9
Extracting NFOR Candidates...
  NFOR candidates: 68
  Total candidates: 77
Two Stage Bayesian Inference...
Keyword Selection...
  Best: FOR_Offset_3_W1 prob=1.0000
Assigning Semantic Clustering...

[1] FOR-NFOR BOUNDARY IDENTIFICATION (Algorithm 1)
  Inferred FOR-NFOR Boundary (B) : 16 bytes
  Execution Time               : 116.0568 seconds
  Ground Truth Boundary        : 12 bytes
  Boundary Absolute Error      : 4 bytes
  Boundary Relative Error      : 33.33%

[2] REGION-PARTITIONED CANDIDATE GENERATION
  FOR Candidates Extracted     : 9
  NFOR Candidates Extracted    : 68
  Total Candidates Evaluated  : 77

[3] TWO-STAGE BAYESIAN INFERENCE RANKINGS (Top Candidates)
-----------------------------------------------------------------
Rank Tag                   Type  p_b